In [ ]:
from pathlib import Path

BASE = Path(r"C:\ai_campus_main_proj\유시아")

matches = list(
    BASE.rglob("daily_inputs_with_kosdaq.csv")
)

print("찾은 파일 수:", len(matches))
for path in matches:
    print(path)

if len(matches) != 1:
    raise ValueError("위 경로를 확인해 분석할 파일 하나를 지정해야 합니다.")

INPUT = matches[0]
ROOT = INPUT.parents[2]

print("\n전달자료 루트:", ROOT)
print("요청서 존재:", (ROOT / "docs/VALIDATION_REQUEST.md").exists())

In [ ]:
import pandas as pd

df = pd.read_csv(
    INPUT,
    parse_dates=["date", "previous_sector_date"]
)

factors = [
    "KR3Y_change", "KR10Y_change", "KR_CORP_AA3Y_change",
    "KR_CREDIT_change", "KR_CURVE_change", "USDKRW_change"
]

cols = [
    "sector_return_pct",
    "kospi_return_pct",
    "kosdaq_return_pct",
    *factors
]

print("전체 행:", len(df), "← 기대값 18,954")
print("업종 수:", df["sector_code"].nunique(), "← 기대값 26")
print("거래일 수:", df["date"].nunique(), "← 기대값 729")
print("기간:", df["date"].min(), "~", df["date"].max())
print("중복:", df.duplicated(["sector_code", "date"]).sum())
print("\n결측 수:")
print(df[cols].isna().sum())
print("\n업종별 표본 수:")
print(df.groupby(["sector_code", "sector_name"]).size())

In [ ]:
import numpy as np

# 1. 원자료 수준값 읽기
wi = pd.read_csv(ROOT / "data/wi26/wi26_daily_history.csv")
fin = pd.read_csv(ROOT / "data/financial/daily_financial_history.csv")
kp = pd.read_csv(ROOT / "data/domestic/kospi_daily_history.csv")
kd = pd.read_csv(ROOT / "data/dual/kosdaq_daily_history.csv")

# 중복된 키가 있으면 계산을 멈춤
assert not wi.duplicated(["sector_code", "date"]).any()
assert not fin.duplicated(["indicator_id", "observation_period"]).any()
assert not kp.duplicated(["date"]).any()
assert not kd.duplicated(["date"]).any()

wi_map = wi.set_index(["sector_code", "date"])["index_value"]
fin_map = fin.set_index(["indicator_id", "observation_period"])["value"]
kp_map = kp.set_index("date")["kospi_index"]
kd_map = kd.set_index("date")["kosdaq_index"]

current = df["date"].dt.strftime("%Y-%m-%d")
previous = df["previous_sector_date"].dt.strftime("%Y-%m-%d")

# 2. 업종별 날짜가 이어지는지 확인
ordered = df.sort_values(["sector_code", "date"])

for code, group in ordered.groupby("sector_code"):
    assert (
        group["previous_sector_date"].iloc[1:].to_numpy()
        == group["date"].iloc[:-1].to_numpy()
    ).all(), f"날짜 연결 불일치: {code}"

assert (df["previous_sector_date"] < df["date"]).all()

# 3. 수준값으로 직접 재계산
recalculated = {
    "sector_return_pct": np.array([
        100 * (wi_map.loc[(code, end)] / wi_map.loc[(code, start)] - 1)
        for code, start, end
        in zip(df["sector_code"], previous, current)
    ]),
    "kospi_return_pct": np.array([
        100 * (kp_map.loc[end] / kp_map.loc[start] - 1)
        for start, end in zip(previous, current)
    ]),
    "kosdaq_return_pct": np.array([
        100 * (kd_map.loc[end] / kd_map.loc[start] - 1)
        for start, end in zip(previous, current)
    ])
}

for column in factors:
    indicator = column.removesuffix("_change")

    before = np.array([
        fin_map.loc[(indicator, day)] for day in previous
    ], dtype=float)

    after = np.array([
        fin_map.loc[(indicator, day)] for day in current
    ], dtype=float)

    if indicator == "USDKRW":
        values = 100 * (after / before - 1)  # 환율: 변화율 %
    elif indicator in ["KR_CREDIT", "KR_CURVE"]:
        values = after - before           # 스프레드 수준값 자체가 bp
    else:
        values = 100 * (after - before)    # 금리 연% 차이 → bp

    recalculated[column] = values

# 4. 기존 입력과 비교
checks = []

for column, calculated in recalculated.items():
    supplied = df[column].to_numpy(dtype=float)
    finite = np.isfinite(calculated) & np.isfinite(supplied)
    matched = finite & np.isclose(
        calculated, supplied, atol=1e-8, rtol=1e-6
    )

    checks.append({
        "column": column,
        "checked_rows": len(supplied),
        "nonfinite_rows": int((~finite).sum()),
        "mismatches": int((~matched).sum()),
        "max_abs_error": float(np.max(np.abs(calculated - supplied)))
    })

audit = pd.DataFrame(checks)
print(audit.to_string(index=False))

# 검증 기록 저장: 노트북을 실행한 폴더 아래 results 생성
OUT = Path.cwd() / "results"
OUT.mkdir(exist_ok=True)
audit.to_csv(
    OUT / "transformation_audit.csv",
    index=False,
    encoding="utf-8-sig"
)

assert audit["mismatches"].sum() == 0, "불일치 원인을 먼저 확인해야 합니다."
print("\n통과: 날짜 연결과 모든 일별 입력 변환이 일치합니다.")
print("저장 위치:", OUT)

In [ ]:
import statsmodels.api as sm

# 1. 분석 대상 고정
CODE = "WI100"
FACTOR = "KR3Y_change"

g = (
    df.loc[df["sector_code"] == CODE]
    .sort_values("date")
    .copy()
)

y = g["sector_return_pct"]
x = g[FACTOR]

print("업종:", g["sector_name"].iloc[0])
print("지표:", FACTOR)
print("표본 수:", len(g))

# 2. 공통 회귀 함수
# 일별 HAC: Bartlett, maxlags=5, 소표본 보정, 정규근사
def fit_daily(y, predictors):
    design = sm.add_constant(predictors, has_constant="add")

    if not np.isfinite(design.to_numpy()).all():
        raise ValueError("설명변수에 결측 또는 무한대가 있습니다.")
    if np.linalg.matrix_rank(design.to_numpy()) != design.shape[1]:
        raise ValueError("설명변수 행렬의 랭크가 부족합니다.")

    return sm.OLS(y, design, missing="raise").fit(
        cov_type="HAC",
        cov_kwds={
            "maxlags": 5,
            "kernel": "bartlett",
            "use_correction": True
        },
        use_t=False
    )

# 3. 상관·편상관
def partial_corr(y, x, controls):
    design = sm.add_constant(controls, has_constant="add")
    y_resid = sm.OLS(y, design, missing="raise").fit().resid
    x_resid = sm.OLS(x, design, missing="raise").fit().resid
    return y_resid.corr(x_resid)

raw_corr = y.corr(x)

partial_kospi = partial_corr(
    y, x, g[["kospi_return_pct"]]
)
partial_dual = partial_corr(
    y, x, g[["kospi_return_pct", "kosdaq_return_pct"]]
)

# 4. 同じ行で基準・追加モデルを推定
models = {
    "factor_only": fit_daily(y, g[[FACTOR]]),
    "kospi_baseline": fit_daily(y, g[["kospi_return_pct"]]),
    "kospi_expanded": fit_daily(
        y, g[["kospi_return_pct", FACTOR]]
    ),
    "dual_baseline": fit_daily(
        y, g[["kospi_return_pct", "kosdaq_return_pct"]]
    ),
    "dual_expanded": fit_daily(
        y, g[["kospi_return_pct", "kosdaq_return_pct", FACTOR]]
    )
}

# 5. 金融指標係数と説明力を一覧化
rows = []

for name, model in models.items():
    row = {
        "model": name,
        "n_obs": int(model.nobs),
        "r2": model.rsquared,
        "adj_r2": model.rsquared_adj
    }

    if FACTOR in model.params.index:
        ci = model.conf_int().loc[FACTOR]
        row.update({
            "beta_factor": model.params[FACTOR],
            "se_hac": model.bse[FACTOR],
            "p_hac": model.pvalues[FACTOR],
            "ci_low": ci.iloc[0],
            "ci_high": ci.iloc[1]
        })

    rows.append(row)

one_result = pd.DataFrame(rows)

# 4. 동일한 행으로 기준모형과 추가모형 추정

# 5. 금융지표 계수와 설명력을 표로 정리

print("\n【상관분석】")
print("단순상관:", raw_corr)
print("KOSPI 통제 후 편상관:", partial_kospi)
print("KOSPI·KOSDAQ 통제 후 편상관:", partial_dual)

print("\n【회귀분석 결과】")
print(one_result.to_string(index=False))

print("\n【금융지표 추가에 따른 수정 R² 증가】")

In [ ]:
# 1. 수정 R² 증가 출력
print("금융지표 추가에 따른 수정 R² 증가")
for control in ["kospi", "dual"]:
    increase = (
        models[f"{control}_expanded"].rsquared_adj
        - models[f"{control}_baseline"].rsquared_adj
    )
    print(f"{control}: {increase:.8f} ({increase * 100:.4f}%p)")

# 2. 재계산을 마쳤으므로 기존 결과 읽기
reference = pd.read_csv(
    ROOT / "reference/dual/model_comparison.csv"
)

selected = reference.loc[
    (reference["sector_code"] == CODE)
    & (reference["factor"] == FACTOR)
    & (reference["horizon_days"] == 1)
    & (reference["mode"] == "same_block")
]

assert len(selected) == 1, "비교 대상이 정확히 1행이어야 합니다."
ref = selected.iloc[0]

total = models["factor_only"]
dual = models["dual_expanded"]

# 3. 같은 의미의 값끼리 비교
actual = {
    "n_obs": len(g),
    "corr_total": raw_corr,
    "partial_corr_kospi_only": partial_kospi,
    "partial_corr_dual": partial_dual,
    "beta_total": total.params[FACTOR],
    "p_total_hac": total.pvalues[FACTOR],
    "total_ci_low": total.conf_int().loc[FACTOR].iloc[0],
    "total_ci_high": total.conf_int().loc[FACTOR].iloc[1],
    "beta_dual": dual.params[FACTOR],
    "p_dual_hac": dual.pvalues[FACTOR],
    "dual_ci_low": dual.conf_int().loc[FACTOR].iloc[0],
    "dual_ci_high": dual.conf_int().loc[FACTOR].iloc[1],
    "p_kospi_only_hac": models["kospi_expanded"].pvalues[FACTOR],
    "r2_total": total.rsquared,
    "r2_dual_market_baseline": models["dual_baseline"].rsquared,
    "r2_dual_expanded": dual.rsquared,
    "delta_adj_r2_dual": (
        dual.rsquared_adj - models["dual_baseline"].rsquared_adj
    )
}

comparison = pd.DataFrame([
    {
        "항목": key,
        "기존값": float(ref[key]),
        "재계산값": value,
        "절대오차": abs(value - float(ref[key])),
        "일치": (
            value == float(ref[key])
            if key == "n_obs"
            else np.isclose(
                value, float(ref[key]), atol=1e-8, rtol=1e-6
            )
        )
    }
    for key, value in actual.items()
])

print("\n기존 결과와의 대조")
print(comparison.to_string(index=False))

comparison.to_csv(
    OUT / "single_pair_reproduction_diff.csv",
    index=False,
    encoding="utf-8-sig"
)

assert comparison["일치"].all(), "불일치 항목을 먼저 확인해야 합니다."
print("\n통과: 비교한 항목이 모두 기존 결과와 일치합니다.")

In [ ]:
# 전체 일별: 26개 업종 × 6개 금융지표
# 다중검정 보정은 다음 단계에서 별도로 확인
results = []

for code, group in df.groupby("sector_code", sort=True):
    group = group.sort_values("date")
    target = group["sector_return_pct"]

    market_kospi = group[["kospi_return_pct"]]
    market_dual = group[["kospi_return_pct", "kosdaq_return_pct"]]

    baseline_kospi = fit_daily(target, market_kospi)
    baseline_dual = fit_daily(target, market_dual)

    for factor in factors:
        indicator = group[factor]

        total = fit_daily(target, group[[factor]])
        expanded_kospi = fit_daily(
            target, group[["kospi_return_pct", factor]]
        )
        expanded_dual = fit_daily(
            target,
            group[["kospi_return_pct", "kosdaq_return_pct", factor]]
        )

        total_ci = total.conf_int().loc[factor]
        dual_ci = expanded_dual.conf_int().loc[factor]

        results.append({
            "sector_code": code,
            "sector_name": group["sector_name"].iloc[0],
            "factor": factor,
            "n_obs": int(expanded_dual.nobs),
            "corr_total": target.corr(indicator),
            "partial_corr_kospi_only": partial_corr(
                target, indicator, market_kospi
            ),
            "partial_corr_dual": partial_corr(
                target, indicator, market_dual
            ),
            "beta_total": total.params[factor],
            "p_total_hac": total.pvalues[factor],
            "total_ci_low": total_ci.iloc[0],
            "total_ci_high": total_ci.iloc[1],
            "beta_dual": expanded_dual.params[factor],
            "se_dual_hac": expanded_dual.bse[factor],
            "p_dual_hac": expanded_dual.pvalues[factor],
            "dual_ci_low": dual_ci.iloc[0],
            "dual_ci_high": dual_ci.iloc[1],
            "beta_kospi": expanded_kospi.params[factor],
            "se_kospi_hac": expanded_kospi.bse[factor],
            "p_kospi_only_hac": expanded_kospi.pvalues[factor],
            "r2_total": total.rsquared,
            "r2_dual_market_baseline": baseline_dual.rsquared,
            "r2_dual_expanded": expanded_dual.rsquared,
            "delta_adj_r2_dual": (
                expanded_dual.rsquared_adj - baseline_dual.rsquared_adj
            )
        })

daily_results = pd.DataFrame(results)
assert len(daily_results) == 156

# 팀원의 일별 결과만 추출
daily_reference = reference.loc[
    (reference["horizon_days"] == 1)
    & (reference["mode"] == "same_block")
].copy()

keys = ["sector_code", "factor"]

assert len(daily_reference) == 156
assert not daily_reference.duplicated(keys).any()

merged = daily_results.merge(
    daily_reference,
    on=keys,
    suffixes=("_new", "_ref"),
    how="outer",
    validate="one_to_one",
    indicator=True
)

assert merged["_merge"].eq("both").all(), "비교 조합이 다릅니다."

compare_columns = [
    "n_obs", "corr_total", "partial_corr_kospi_only",
    "partial_corr_dual", "beta_total", "p_total_hac",
    "total_ci_low", "total_ci_high", "beta_dual",
    "p_dual_hac", "dual_ci_low", "dual_ci_high",
    "p_kospi_only_hac", "r2_total",
    "r2_dual_market_baseline", "r2_dual_expanded",
    "delta_adj_r2_dual"
]

differences = []

for _, row in merged.iterrows():
    for column in compare_columns:
        new = float(row[f"{column}_new"])
        old = float(row[f"{column}_ref"])

        differences.append({
            "sector_code": row["sector_code"],
            "factor": row["factor"],
            "item": column,
            "reference_value": old,
            "recalculated_value": new,
            "absolute_error": abs(new - old),
            "matched": (
                new == old if column == "n_obs"
                else bool(np.isclose(new, old, atol=1e-8, rtol=1e-6))
            )
        })

daily_diff = pd.DataFrame(differences)

daily_results.to_csv(
    OUT / "daily_recalculated_results.csv",
    index=False, encoding="utf-8-sig"
)
daily_diff.to_csv(
    OUT / "daily_reproduction_diff.csv",
    index=False, encoding="utf-8-sig"
)

print("분석 조합 수:", len(daily_results))
print("비교 항목 수:", len(daily_diff))
print("불일치 수:", int((~daily_diff["matched"]).sum()))
print("최대 절대오차:", daily_diff["absolute_error"].max())

if daily_diff["matched"].all():
    print("통과: 전체 일별 156개 조합의 비교 항목이 일치합니다.")
else:
    print("\n불일치 항목:")
    print(
        daily_diff.loc[~daily_diff["matched"]]
        .head(20).to_string(index=False)
    )

In [ ]:
from statsmodels.stats.multitest import multipletests

# 최초 KOSPI 통제 일별 결과: 156개 검정
initial_reference = pd.read_csv(
    ROOT / "reference/initial/regression_comparison.csv"
)

keys = ["sector_code", "factor"]

assert len(initial_reference) == 156
assert not initial_reference.duplicated(keys).any()

# 팀원의 최초 분석과 동일하게 전체 156개 p값에 BY 보정
# 이 값은 종합 결과의 BY1560/BY2340과 별개
daily_results["p_kospi_by156"] = multipletests(
    daily_results["p_kospi_only_hac"].to_numpy(),
    alpha=0.05,
    method="fdr_by"
)[1]

check = daily_results.merge(
    initial_reference[
        keys + [
            "beta_factor", "se_factor_hac",
            "p_factor_hac", "p_factor_by"
        ]
    ],
    on=keys,
    how="outer",
    validate="one_to_one",
    indicator=True
)

assert check["_merge"].eq("both").all()

mapping = {
    "beta_kospi": "beta_factor",
    "se_kospi_hac": "se_factor_hac",
    "p_kospi_only_hac": "p_factor_hac",
    "p_kospi_by156": "p_factor_by"
}

records = []

for _, row in check.iterrows():
    for new_column, reference_column in mapping.items():
        new_value = float(row[new_column])
        reference_value = float(row[reference_column])

        records.append({
            "sector_code": row["sector_code"],
            "factor": row["factor"],
            "item": new_column,
            "reference_value": reference_value,
            "recalculated_value": new_value,
            "absolute_error": abs(new_value - reference_value),
            "matched": bool(np.isclose(
                new_value, reference_value,
                atol=1e-8, rtol=1e-6
            ))
        })

kospi_diff = pd.DataFrame(records)

kospi_diff.to_csv(
    OUT / "daily_kospi_by156_reproduction_diff.csv",
    index=False,
    encoding="utf-8-sig"
)

daily_results.to_csv(
    OUT / "daily_recalculated_results.csv",
    index=False,
    encoding="utf-8-sig"
)

print("추가 비교 항목 수:", len(kospi_diff))
print("불일치 수:", int((~kospi_diff["matched"]).sum()))
print("최대 절대오차:", kospi_diff["absolute_error"].max())

print(
    "KOSPI 통제: 보정 전 p < 0.05 조합 수:",
    int((daily_results["p_kospi_only_hac"] < 0.05).sum())
)
print(
    "KOSPI 통제: BY156 보정 후 p < 0.05 조합 수:",
    int((daily_results["p_kospi_by156"] < 0.05).sum())
)

assert kospi_diff["matched"].all(), "불일치 원인을 확인해야 합니다."
print("\n통과: KOSPI 계수·HAC 표준오차·p값·BY156 보정이 일치합니다.")

In [ ]:
# 1. 결과를 보기 전에 시간 분할 기준 고정
split_plan = pd.DataFrame([
    {
        "fold": 1,
        "train_start": "2023-11-07",
        "train_end": "2025-05-05",
        "test_start": "2025-05-06",
        "test_end": "2025-11-05"
    },
    {
        "fold": 2,
        "train_start": "2023-11-07",
        "train_end": "2025-11-05",
        "test_start": "2025-11-06",
        "test_end": "2026-05-05"
    },
    {
        "fold": 3,
        "train_start": "2023-11-07",
        "train_end": "2026-05-05",
        "test_start": "2026-05-06",
        "test_end": "2026-11-05"
    }
])

date_columns = [
    "train_start", "train_end", "test_start", "test_end"
]
for column in date_columns:
    split_plan[column] = pd.to_datetime(split_plan[column])

# 2. 모든 업종이 같은 거래일을 갖는지 확인
trading_dates = pd.DatetimeIndex(
    sorted(df["date"].unique())
)

for code, group in df.groupby("sector_code"):
    sector_dates = pd.DatetimeIndex(
        group.sort_values("date")["date"]
    )
    assert sector_dates.equals(trading_dates), (
        f"업종별 거래일 불일치: {code}"
    )

# 3. 실제 존재하는 거래일만 선택
split_rows = []
cv_splits = {}
previous_test_dates = set()

for row in split_plan.itertuples(index=False):
    train_dates = trading_dates[
        (trading_dates >= row.train_start)
        & (trading_dates <= row.train_end)
    ]
    test_dates = trading_dates[
        (trading_dates >= row.test_start)
        & (trading_dates <= row.test_end)
    ]

    assert len(train_dates) > 0 and len(test_dates) > 0
    assert train_dates.max() < test_dates.min()
    assert set(train_dates).isdisjoint(set(test_dates))

    # fold 간 검증 기간은 서로 겹치지 않아야 함
    assert previous_test_dates.isdisjoint(set(test_dates))
    previous_test_dates.update(test_dates)

    cv_splits[row.fold] = {
        "train_dates": train_dates,
        "test_dates": test_dates
    }

    for period, dates, planned_start, planned_end in [
        ("train", train_dates, row.train_start, row.train_end),
        ("test", test_dates, row.test_start, row.test_end)
    ]:
        selected = df.loc[df["date"].isin(dates)]
        counts = selected.groupby("sector_code").size()

        assert len(counts) == 26
        assert counts.eq(len(dates)).all()

        split_rows.append({
            "fold": row.fold,
            "period": period,
            "planned_start": planned_start.date().isoformat(),
            "planned_end": planned_end.date().isoformat(),
            "actual_start": dates.min().date().isoformat(),
            "actual_end": dates.max().date().isoformat(),
            "trading_days_per_sector": len(dates),
            "sector_count": len(counts),
            "total_rows": len(selected)
        })

splits = pd.DataFrame(split_rows)

# 4. 분할 기준과 실제 표본 기록 저장
split_plan.to_csv(
    OUT / "split_plan.csv",
    index=False,
    date_format="%Y-%m-%d",
    encoding="utf-8-sig"
)

splits.to_csv(
    OUT / "splits.csv",
    index=False,
    encoding="utf-8-sig"
)

print(splits.to_string(index=False))
print("\n통과: 학습·검증 순서와 업종별 거래일이 일치합니다.")
print("통과: fold 간 검증 기간이 겹치지 않습니다.")

In [ ]:
# KOSPI 단독 및 KOSPI·KOSDAQ 통제를 각각 평가
control_sets = {
    "kospi": ["kospi_return_pct"],
    "dual": ["kospi_return_pct", "kosdaq_return_pct"]
}

metric_rows = []
fitted_frames = []
coefficient_rows = []


def error_metrics(actual, predicted):
    error = np.asarray(actual) - np.asarray(predicted)
    mse = float(np.mean(error ** 2))

    return {
        "mse": mse,
        "rmse": float(np.sqrt(mse)),
        "mae": float(np.mean(np.abs(error)))
    }


for fold, dates in cv_splits.items():
    for code, group in df.groupby("sector_code", sort=True):
        group = group.sort_values("date")

        train = group.loc[
            group["date"].isin(dates["train_dates"])
        ].copy()

        test = group.loc[
            group["date"].isin(dates["test_dates"])
        ].copy()

        assert len(train) == len(dates["train_dates"])
        assert len(test) == len(dates["test_dates"])
        assert train["date"].max() < test["date"].min()

        for control, market_columns in control_sets.items():
            # 기준모형은 학습 구간에서만 추정
            baseline = fit_daily(
                train["sector_return_pct"],
                train[market_columns]
            )

            baseline_test_design = sm.add_constant(
                test[market_columns],
                has_constant="add"
            ).reindex(columns=baseline.model.exog_names)

            # 학습한 계수를 검증 구간에 고정 적용
            baseline_pred = baseline.predict(baseline_test_design)

            baseline_errors = error_metrics(
                test["sector_return_pct"],
                baseline_pred
            )

            for factor in factors:
                predictors = market_columns + [factor]

                # 추가모형도 기준모형과 같은 학습 행으로 추정
                expanded = fit_daily(
                    train["sector_return_pct"],
                    train[predictors]
                )

                assert baseline.nobs == expanded.nobs == len(train)

                expanded_test_design = sm.add_constant(
                    test[predictors],
                    has_constant="add"
                ).reindex(columns=expanded.model.exog_names)

                expanded_pred = expanded.predict(expanded_test_design)

                assert np.isfinite(baseline_pred).all()
                assert np.isfinite(expanded_pred).all()

                expanded_errors = error_metrics(
                    test["sector_return_pct"],
                    expanded_pred
                )

                improvement = (
                    100 * (
                        1 - expanded_errors["mse"]
                        / baseline_errors["mse"]
                    )
                    if baseline_errors["mse"] > 0
                    else np.nan
                )

                model_id = f"{code}__{factor}__{control}"

                metric_rows.append({
                    "model_id": model_id,
                    "fold": fold,
                    "sector_code": code,
                    "sector_name": group["sector_name"].iloc[0],
                    "factor": factor,
                    "control": control,
                    "horizon_days": 1,
                    "mode": "same_block",
                    "n_train": len(train),
                    "n_test": len(test),
                    "train_start": train["date"].min(),
                    "train_end": train["date"].max(),
                    "test_start": test["date"].min(),
                    "test_end": test["date"].max(),
                    "mse_baseline": baseline_errors["mse"],
                    "mse_expanded": expanded_errors["mse"],
                    "rmse_baseline": baseline_errors["rmse"],
                    "rmse_expanded": expanded_errors["rmse"],
                    "mae_baseline": baseline_errors["mae"],
                    "mae_expanded": expanded_errors["mae"],
                    "mse_improvement_pct": improvement
                })

                fitted_frames.append(pd.DataFrame({
                    "model_id": model_id,
                    "fold": fold,
                    "date": test["date"].to_numpy(),
                    "sector_code": code,
                    "factor": factor,
                    "control": control,
                    "actual": test["sector_return_pct"].to_numpy(),
                    "pred_baseline": baseline_pred.to_numpy(),
                    "pred_expanded": expanded_pred.to_numpy(),
                    "error_baseline": (
                        test["sector_return_pct"].to_numpy()
                        - baseline_pred.to_numpy()
                    ),
                    "error_expanded": (
                        test["sector_return_pct"].to_numpy()
                        - expanded_pred.to_numpy()
                    )
                }))

                # 학습 구간별 금융지표 계수와 HAC 추정 결과 기록
                ci = expanded.conf_int().loc[factor]

                coefficient_rows.append({
                    "model_id": model_id,
                    "fold": fold,
                    "sector_code": code,
                    "factor": factor,
                    "control": control,
                    "n_train": len(train),
                    "beta_factor": expanded.params[factor],
                    "se_hac": expanded.bse[factor],
                    "ci_low": ci.iloc[0],
                    "ci_high": ci.iloc[1],
                    "p_hac": expanded.pvalues[factor],
                    "p_adjustment": "미보정·기술 목적",
                    "hac_maxlags": 5
                })

    print(f"분할 {fold} 완료")


cv_metrics = pd.DataFrame(metric_rows)
cv_fitted = pd.concat(fitted_frames, ignore_index=True)
coefficient_stability = pd.DataFrame(coefficient_rows)

assert len(cv_metrics) == 936
assert not cv_metrics.duplicated(["model_id", "fold"]).any()
assert not cv_fitted.duplicated(
    ["model_id", "fold", "date"]
).any()

# 평가 결과·검증일별 계산값·학습 계수 저장
tables = {
    "cv_metrics.csv": cv_metrics,
    "cv_fitted_values.csv": cv_fitted,
    "coefficient_stability.csv": coefficient_stability
}

for filename, table in tables.items():
    table.to_csv(
        OUT / filename,
        index=False,
        encoding="utf-8-sig"
    )

# 통제 방식과 분할별 전체 156개 조합의 결과 요약
summary = (
    cv_metrics.groupby(["control", "fold"])
    .agg(
        evaluated_pairs=("model_id", "size"),
        improved_pairs=(
            "mse_improvement_pct",
            lambda values: int((values > 0).sum())
        ),
        median_improvement_pct=("mse_improvement_pct", "median"),
        mean_improvement_pct=("mse_improvement_pct", "mean"),
        worst_improvement_pct=("mse_improvement_pct", "min")
    )
    .reset_index()
)

summary.to_csv(
    OUT / "cv_fold_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n검증 결과:")
print(summary.to_string(index=False))
print("\n저장 위치:", OUT)

In [ ]:
# 업종·금융지표·시장 통제 방식별로 3개 분할 결과 통합
keys = ["sector_code", "sector_name", "factor", "control"]
pair_rows = []

for key, group in cv_metrics.groupby(keys, sort=True):
    assert len(group) == 3
    assert set(group["fold"]) == {1, 2, 3}

    n_total = int(group["n_test"].sum())

    # 분할별 오차 합을 합산해 전체 검증 기간의 오차 계산
    # 분할마다 표본 수가 다르므로 표본 수로 가중
    pooled_mse_baseline = float(
        (group["mse_baseline"] * group["n_test"]).sum() / n_total
    )
    pooled_mse_expanded = float(
        (group["mse_expanded"] * group["n_test"]).sum() / n_total
    )

    pooled_mae_baseline = float(
        (group["mae_baseline"] * group["n_test"]).sum() / n_total
    )
    pooled_mae_expanded = float(
        (group["mae_expanded"] * group["n_test"]).sum() / n_total
    )

    pooled_improvement = (
        100 * (1 - pooled_mse_expanded / pooled_mse_baseline)
        if pooled_mse_baseline > 0 else np.nan
    )

    row = dict(zip(keys, key))
    row.update({
        "n_test_total": n_total,
        "improved_folds": int(
            (group["mse_improvement_pct"] > 0).sum()
        ),
        "worst_fold_improvement_pct": (
            group["mse_improvement_pct"].min()
        ),
        "pooled_mse_baseline": pooled_mse_baseline,
        "pooled_mse_expanded": pooled_mse_expanded,
        "pooled_rmse_baseline": np.sqrt(pooled_mse_baseline),
        "pooled_rmse_expanded": np.sqrt(pooled_mse_expanded),
        "pooled_mae_baseline": pooled_mae_baseline,
        "pooled_mae_expanded": pooled_mae_expanded,
        "pooled_mse_improvement_pct": pooled_improvement
    })

    for fold in [1, 2, 3]:
        row[f"fold{fold}_improvement_pct"] = float(
            group.loc[
                group["fold"] == fold, "mse_improvement_pct"
            ].iloc[0]
        )

    pair_rows.append(row)

cv_pair_summary = pd.DataFrame(pair_rows)
assert len(cv_pair_summary) == 312

# 모든 조합의 통합 결과 저장
cv_pair_summary.to_csv(
    OUT / "cv_pair_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

# 시장 통제 방식별 전체 결과 요약
overview_rows = []

for control, group in cv_pair_summary.groupby("control"):
    overview_rows.append({
        "control": control,
        "total_pairs": len(group),
        "pooled_improved_pairs": int(
            (group["pooled_mse_improvement_pct"] > 0).sum()
        ),
        "all_3_folds_improved": int(
            (group["improved_folds"] == 3).sum()
        ),
        "pooled_median_improvement_pct": (
            group["pooled_mse_improvement_pct"].median()
        ),
        "pooled_mean_improvement_pct": (
            group["pooled_mse_improvement_pct"].mean()
        )
    })

overview = pd.DataFrame(overview_rows)

overview.to_csv(
    OUT / "cv_overall_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

# 출력할 때만 열 이름을 한국어로 표시
overview_labels = {
    "control": "통제방식",
    "total_pairs": "전체조합수",
    "pooled_improved_pairs": "통합오차개선조합수",
    "all_3_folds_improved": "3개기간모두개선조합수",
    "pooled_median_improvement_pct": "통합개선율중앙값(%)",
    "pooled_mean_improvement_pct": "통합개선율평균(%)"
}

print("【전체 검증 기간의 통합 결과】")
print(
    overview.rename(columns=overview_labels)
    .replace({"통제방식": {
        "kospi": "KOSPI",
        "dual": "KOSPI·KOSDAQ"
    }})
    .to_string(index=False)
)

# 3개 기간 모두 개선된 조합 표시
stable = cv_pair_summary.loc[
    cv_pair_summary["improved_folds"] == 3
].sort_values(
    ["control", "pooled_mse_improvement_pct"],
    ascending=[True, False]
)

display_columns = [
    "sector_name", "factor", "control",
    "fold1_improvement_pct", "fold2_improvement_pct",
    "fold3_improvement_pct", "pooled_mse_improvement_pct"
]

detail_labels = {
    "sector_name": "업종",
    "factor": "금융지표",
    "control": "통제방식",
    "fold1_improvement_pct": "분할1개선율(%)",
    "fold2_improvement_pct": "분할2개선율(%)",
    "fold3_improvement_pct": "분할3개선율(%)",
    "pooled_mse_improvement_pct": "통합개선율(%)"
}

print("\n【3개 기간 모두 개선된 조합】")

if stable.empty:
    print("해당 조합 없음")
else:
    print(
        stable[display_columns]
        .rename(columns=detail_labels)
        .to_string(index=False)
    )

print("\n저장 위치:", OUT)

In [ ]:
print(
    stable.loc[stable["control"] == "kospi", display_columns]
    .rename(columns=detail_labels)
    .to_string(index=False)
)


In [ ]:
# 1. 분할별 금융지표 계수 확인
coefs = coefficient_stability.copy()

assert not coefs.duplicated(["model_id", "fold"]).any()
assert coefs.groupby("model_id").size().eq(3).all()

required = ["beta_factor", "se_hac", "ci_low", "ci_high", "p_hac"]
assert np.isfinite(coefs[required].to_numpy()).all()

# 계수의 부호와 개별 95% 신뢰구간의 0 포함 여부
coefs["coefficient_sign"] = np.sign(coefs["beta_factor"]).astype(int)
coefs["ci_includes_zero"] = (
    (coefs["ci_low"] <= 0)
    & (coefs["ci_high"] >= 0)
)

# 2. 각 조합의 분할별 계수를 한 행으로 정리
stability_rows = []

for model_id, group in coefs.groupby("model_id", sort=True):
    group = group.sort_values("fold")
    assert set(group["fold"]) == {1, 2, 3}

    signs = group["coefficient_sign"].to_numpy()
    coefficients = group["beta_factor"].to_numpy()

    row = {
        "model_id": model_id,
        "sector_code": group["sector_code"].iloc[0],
        "factor": group["factor"].iloc[0],
        "control": group["control"].iloc[0],

        # 세 계수가 모두 양수이거나 모두 음수인지 확인
        "same_nonzero_sign": bool(
            (signs > 0).all() or (signs < 0).all()
        ),
        "sign_change_count": int(
            np.sum(signs[1:] != signs[:-1])
        ),
        "beta_min": float(coefficients.min()),
        "beta_max": float(coefficients.max()),
        "beta_range": float(
            coefficients.max() - coefficients.min()
        ),
        "ci_includes_zero_folds": int(
            group["ci_includes_zero"].sum()
        )
    }

    for fold in [1, 2, 3]:
        part = group.loc[group["fold"] == fold].iloc[0]
        row[f"beta_fold{fold}"] = part["beta_factor"]
        row[f"ci_low_fold{fold}"] = part["ci_low"]
        row[f"ci_high_fold{fold}"] = part["ci_high"]

    stability_rows.append(row)

stability_table = pd.DataFrame(stability_rows)

# 3. 전체 기간에서 재현한 계수와 연결
# 전체 기간 계수는 비교용이며 교차검증 계산에는 사용하지 않음
full_period = daily_results[
    ["sector_code", "factor", "beta_kospi", "beta_dual"]
].melt(
    id_vars=["sector_code", "factor"],
    value_vars=["beta_kospi", "beta_dual"],
    var_name="control",
    value_name="beta_full_period"
)

full_period["control"] = full_period["control"].map({
    "beta_kospi": "kospi",
    "beta_dual": "dual"
})

join_keys = ["sector_code", "factor", "control"]

stability_table = stability_table.merge(
    full_period,
    on=join_keys,
    how="left",
    validate="one_to_one"
)

assert stability_table["beta_full_period"].notna().all()

# 4. 앞서 계산한 검증 오차 결과와 연결
review_table = cv_pair_summary.merge(
    stability_table,
    on=join_keys,
    how="outer",
    validate="one_to_one",
    indicator=True
)

assert review_table["_merge"].eq("both").all()
review_table = review_table.drop(columns="_merge")
assert len(review_table) == 312

# MSE와 별도로 MAE도 개선됐는지 확인
review_table["mae_improvement_pct"] = np.where(
    review_table["pooled_mae_baseline"] > 0,
    100 * (
        1 - review_table["pooled_mae_expanded"]
        / review_table["pooled_mae_baseline"]
    ),
    np.nan
)

# 전체 조합 저장: 불리한 결과도 포함
review_table.to_csv(
    OUT / "cv_coefficient_review.csv",
    index=False,
    encoding="utf-8-sig"
)

# 5. 통제 방식별 계수 방향 요약
print("【전체 조합의 계수 방향 점검】")

for control, group in review_table.groupby("control"):
    label = "KOSPI" if control == "kospi" else "KOSPI·KOSDAQ"

    print(f"\n통제 방식: {label}")
    print("전체 조합 수:", len(group))
    print(
        "세 학습 기간에서 계수 방향이 같은 조합 수:",
        int(group["same_nonzero_sign"].sum())
    )
    print(
        "세 검증 기간 모두 개선되고 계수 방향도 같은 조합 수:",
        int((
            (group["improved_folds"] == 3)
            & group["same_nonzero_sign"]
        ).sum())
    )

# 6. 세 검증 기간 모두 개선된 조합의 상세 내용 출력
selected = review_table.loc[
    review_table["improved_folds"] == 3
].sort_values(
    ["control", "pooled_mse_improvement_pct"],
    ascending=[True, False]
)

columns = [
    "sector_name", "factor", "control",
    "beta_full_period",
    "beta_fold1", "beta_fold2", "beta_fold3",
    "same_nonzero_sign", "ci_includes_zero_folds",
    "pooled_mse_improvement_pct", "mae_improvement_pct"
]

labels = {
    "sector_name": "업종",
    "factor": "금융지표",
    "control": "통제방식",
    "beta_full_period": "전체기간계수",
    "beta_fold1": "학습1계수",
    "beta_fold2": "학습2계수",
    "beta_fold3": "학습3계수",
    "same_nonzero_sign": "계수방향일치",
    "ci_includes_zero_folds": "신뢰구간0포함횟수",
    "pooled_mse_improvement_pct": "통합MSE개선율(%)",
    "mae_improvement_pct": "통합MAE개선율(%)"
}

print("\n【세 검증 기간 모두 개선된 조합의 계수 점검】")
print(
    selected[columns]
    .rename(columns=labels)
    .to_string(index=False)
)

print("\n저장 위치:", OUT / "cv_coefficient_review.csv")


In [ ]:
# 검증일별 오차로 통합 성적 재확인 및 개선 집중도 점검
diagnostic_rows = []

for model_id, group in cv_fitted.groupby("model_id", sort=True):
    group = group.sort_values("date").copy()

    error_baseline = group["error_baseline"].to_numpy()
    error_expanded = group["error_expanded"].to_numpy()

    # 양수이면 해당 날짜의 제곱오차가 감소
    daily_gain = error_baseline**2 - error_expanded**2

    mse_baseline = float(np.mean(error_baseline**2))
    mse_expanded = float(np.mean(error_expanded**2))

    # 개선된 날짜의 이득 중 상위 5개 날짜가 차지하는 비중
    # 악화된 날짜의 손실을 차감하기 전의 '양의 이득' 기준
    positive_gain = daily_gain[daily_gain > 0]
    positive_gain_sum = float(positive_gain.sum())

    top5_share = (
        100 * np.sort(positive_gain)[-5:].sum() / positive_gain_sum
        if positive_gain_sum > 0 else np.nan
    )

    diagnostic_rows.append({
        "model_id": model_id,
        "n_test": len(group),
        "mse_baseline_from_errors": mse_baseline,
        "mse_expanded_from_errors": mse_expanded,
        "improved_day_pct": 100 * float(
            (np.abs(error_expanded) < np.abs(error_baseline)).mean()
        ),
        "top5_positive_gain_share_pct": top5_share,
        "max_abs_error_baseline": float(
            np.max(np.abs(error_baseline))
        ),
        "max_abs_error_expanded": float(
            np.max(np.abs(error_expanded))
        )
    })

error_diagnostics = pd.DataFrame(diagnostic_rows)

# 앞서 통합한 MSE와 일별 오차에서 직접 계산한 MSE 대조
error_review = review_table.merge(
    error_diagnostics,
    on="model_id",
    validate="one_to_one"
)

assert len(error_review) == 312

assert np.allclose(
    error_review["pooled_mse_baseline"],
    error_review["mse_baseline_from_errors"],
    atol=1e-8, rtol=1e-6
)
assert np.allclose(
    error_review["pooled_mse_expanded"],
    error_review["mse_expanded_from_errors"],
    atol=1e-8, rtol=1e-6
)

error_review.to_csv(
    OUT / "cv_error_diagnostics.csv",
    index=False,
    encoding="utf-8-sig"
)

# 전체 조합에서 제곱오차 변화가 큰 날짜를 각각 5개씩 기록
date_rows = []

for model_id, group in cv_fitted.groupby("model_id", sort=True):
    group = group.copy()
    group["squared_error_gain"] = (
        group["error_baseline"]**2 - group["error_expanded"]**2
    )

    largest_changes = group.loc[
        group["squared_error_gain"].abs().nlargest(5).index
    ].copy()

    date_rows.append(largest_changes)

pd.concat(date_rows, ignore_index=True).to_csv(
    OUT / "largest_error_change_dates.csv",
    index=False,
    encoding="utf-8-sig"
)

# 세 기간 모두 개선된 조합만 화면에 표시
show = error_review.loc[
    error_review["improved_folds"] == 3
].sort_values(
    ["control", "pooled_mse_improvement_pct"],
    ascending=[True, False]
)

columns = [
    "sector_name", "factor", "control",
    "pooled_mse_improvement_pct", "mae_improvement_pct",
    "improved_day_pct", "top5_positive_gain_share_pct",
    "max_abs_error_baseline", "max_abs_error_expanded"
]

labels = {
    "sector_name": "업종",
    "factor": "금융지표",
    "control": "통제방식",
    "pooled_mse_improvement_pct": "MSE개선율(%)",
    "mae_improvement_pct": "MAE개선율(%)",
    "improved_day_pct": "절대오차개선일비율(%)",
    "top5_positive_gain_share_pct": "양의이득상위5일비중(%)",
    "max_abs_error_baseline": "기준최대절대오차(%p)",
    "max_abs_error_expanded": "추가최대절대오차(%p)"
}

print("통과: 일별 오차로 재계산한 MSE가 기존 통합 결과와 일치합니다.")
print("\n【세 기간 모두 개선된 조합의 오차 점검】")
print(show[columns].rename(columns=labels).to_string(index=False))

In [ ]:
from statsmodels.stats.diagnostic import acorr_ljungbox, het_breuschpagan
from statsmodels.stats.outliers_influence import variance_inflation_factor

diagnostic_rows = []
vif_rows = []
influence_rows = []

for fold, dates in cv_splits.items():
    for code, group in df.groupby("sector_code", sort=True):
        train = (
            group.loc[group["date"].isin(dates["train_dates"])]
            .sort_values("date")
            .copy()
        )

        for control, market_columns in control_sets.items():
            for factor in factors:
                model_id = f"{code}__{factor}__{control}"

                design = sm.add_constant(
                    train[market_columns + [factor]],
                    has_constant="add"
                )

                # 진단용 일반 OLS 적합
                # 기존 HAC 모형과 점추정·잔차는 같고 추론 방식이 다름
                model = sm.OLS(
                    train["sector_return_pct"],
                    design,
                    missing="raise"
                ).fit()

                residuals = model.resid.to_numpy()

                # 잔차 자기상관: 5시차까지의 Ljung–Box 진단
                lb = acorr_ljungbox(
                    residuals,
                    lags=[5],
                    return_df=True
                )

                # 이분산: 정규성 가정을 완화한 Koenker 방식
                bp_lm, bp_p, _, _ = het_breuschpagan(
                    residuals,
                    design,
                    robust=True
                )

                influence = model.get_influence()
                cooks = influence.cooks_distance[0]
                leverage = influence.hat_matrix_diag

                # 절편을 제외한 모든 설명변수의 VIF 기록
                model_vifs = []

                for position, name in enumerate(design.columns):
                    if name == "const":
                        continue

                    value = float(variance_inflation_factor(
                        design.to_numpy(), position
                    ))
                    model_vifs.append(value)

                    vif_rows.append({
                        "model_id": model_id,
                        "fold": fold,
                        "variable": name,
                        "vif": value
                    })

                # 단위 차이를 줄인 설명변수 조건수도 별도 기록
                predictors = design.drop(columns="const")
                scales = predictors.std(ddof=0)

                if (scales == 0).any():
                    raise ValueError("상수인 설명변수가 있습니다.")

                standardized = (
                    predictors - predictors.mean()
                ) / scales

                scaled_design = sm.add_constant(
                    standardized,
                    has_constant="add"
                )

                diagnostic_rows.append({
                    "model_id": model_id,
                    "fold": fold,
                    "sector_code": code,
                    "factor": factor,
                    "control": control,
                    "n_train": len(train),
                    "rank": np.linalg.matrix_rank(design.to_numpy()),
                    "n_parameters": design.shape[1],
                    "condition_number_raw": np.linalg.cond(
                        design.to_numpy()
                    ),
                    "condition_number_scaled": np.linalg.cond(
                        scaled_design.to_numpy()
                    ),
                    "max_vif": max(model_vifs),
                    "ljung_box_lag5_p": float(lb["lb_pvalue"].iloc[0]),
                    "koenker_lm": float(bp_lm),
                    "koenker_p": float(bp_p),
                    "max_cooks_distance": float(cooks.max()),
                    "max_leverage": float(leverage.max())
                })

                # Cook 거리 상위 3개 학습 관측값 기록
                # 표시만 하며 삭제하지 않음
                for position in np.argsort(cooks)[-3:][::-1]:
                    influence_rows.append({
                        "model_id": model_id,
                        "fold": fold,
                        "date": train["date"].iloc[position],
                        "cooks_distance": float(cooks[position]),
                        "leverage": float(leverage[position]),
                        "residual": float(residuals[position])
                    })

    print(f"분할 {fold} 진단 완료")

model_diagnostics = pd.DataFrame(diagnostic_rows)
full_model_vif = pd.DataFrame(vif_rows)
influential_dates = pd.DataFrame(influence_rows)

assert len(model_diagnostics) == 936
assert (
    model_diagnostics["rank"]
    == model_diagnostics["n_parameters"]
).all()

for filename, table in {
    "model_diagnostics.csv": model_diagnostics,
    "full_model_vif.csv": full_model_vif,
    "influential_training_dates.csv": influential_dates
}.items():
    table.to_csv(
        OUT / filename,
        index=False,
        encoding="utf-8-sig"
    )

# 진단 p값은 다중검정 미보정이며 참고용
diagnostic_summary = (
    model_diagnostics.groupby("control")
    .agg(
        model_count=("model_id", "size"),
        max_vif=("max_vif", "max"),
        max_scaled_condition=("condition_number_scaled", "max"),
        autocorrelation_flags=(
            "ljung_box_lag5_p",
            lambda values: int((values < 0.05).sum())
        ),
        heteroskedasticity_flags=(
            "koenker_p",
            lambda values: int((values < 0.05).sum())
        ),
        max_cooks_distance=("max_cooks_distance", "max")
    )
    .reset_index()
)

print("\n【학습 모형 진단 요약】")
print(
    diagnostic_summary.rename(columns={
        "control": "통제방식",
        "model_count": "모형수",
        "max_vif": "최대VIF",
        "max_scaled_condition": "최대표준화조건수",
        "autocorrelation_flags": "자기상관참고표시수",
        "heteroskedasticity_flags": "이분산참고표시수",
        "max_cooks_distance": "최대Cook거리"
    }).to_string(index=False)
)

print("\n주의: 진단 p값은 미보정 참고값이며 자동 합격·탈락 기준이 아닙니다.")

In [ ]:
# 1. 영향 관측값에 업종·지표·학습 기간 정보 연결
model_info = cv_metrics[
    [
        "model_id", "fold", "sector_code", "sector_name",
        "factor", "control", "train_start", "train_end"
    ]
].copy()

assert not model_info.duplicated(["model_id", "fold"]).any()

influence_detail = influential_dates.merge(
    model_info,
    on=["model_id", "fold"],
    how="left",
    validate="many_to_one"
)

assert influence_detail["sector_code"].notna().all()

# 원래 입력값도 연결해 실제 큰 변동일인지 확인
source_columns = [
    "sector_code", "date", "sector_return_pct",
    "kospi_return_pct", "kosdaq_return_pct", *factors
]

influence_detail = influence_detail.merge(
    df[source_columns],
    on=["sector_code", "date"],
    how="left",
    validate="many_to_one"
)

# 각 행에서 해당 금융지표의 실제 변화량 추출
influence_detail["factor_change"] = [
    row[factor]
    for factor, (_, row) in zip(
        influence_detail["factor"],
        influence_detail.iterrows()
    )
]

influence_detail.to_csv(
    OUT / "influential_dates_with_inputs.csv",
    index=False,
    encoding="utf-8-sig"
)

# 2. 전체 모형에서 Cook 거리가 큰 사례 10개
columns = [
    "sector_name", "factor", "control", "fold", "date",
    "cooks_distance", "leverage", "residual",
    "sector_return_pct", "kospi_return_pct",
    "kosdaq_return_pct", "factor_change"
]

labels = {
    "sector_name": "업종",
    "factor": "금융지표",
    "control": "통제방식",
    "fold": "분할",
    "date": "날짜",
    "cooks_distance": "Cook거리",
    "leverage": "레버리지",
    "residual": "학습잔차(%p)",
    "sector_return_pct": "업종수익률(%)",
    "kospi_return_pct": "KOSPI수익률(%)",
    "kosdaq_return_pct": "KOSDAQ수익률(%)",
    "factor_change": "금융지표변화"
}

print("【전체 모형의 Cook 거리 상위 10개 사례】")
print(
    influence_detail.nlargest(10, "cooks_distance")[columns]
    .rename(columns=labels)
    .to_string(index=False)
)

# 3. 각 조합의 세 학습 기간 중 최대 Cook 거리
pair_diagnostics = (
    model_diagnostics.groupby("model_id")
    .agg(
        max_cooks_distance=("max_cooks_distance", "max"),
        max_vif=("max_vif", "max")
    )
    .reset_index()
)

final_review = review_table.merge(
    pair_diagnostics,
    on="model_id",
    how="left",
    validate="one_to_one"
)

assert len(final_review) == 312

final_review.to_csv(
    OUT / "daily_validation_review.csv",
    index=False,
    encoding="utf-8-sig"
)

show = final_review.loc[
    final_review["improved_folds"] == 3
].sort_values(
    ["control", "pooled_mse_improvement_pct"],
    ascending=[True, False]
)

print("\n【세 기간 모두 개선된 조합의 영향 관측값 점검】")
print(
    show[
        [
            "sector_name", "factor", "control",
            "pooled_mse_improvement_pct",
            "max_cooks_distance", "max_vif"
        ]
    ]
    .rename(columns={
        "sector_name": "업종",
        "factor": "금융지표",
        "control": "통제방식",
        "pooled_mse_improvement_pct": "통합MSE개선율(%)",
        "max_cooks_distance": "최대Cook거리",
        "max_vif": "최대VIF"
    })
    .to_string(index=False)
)

print("\n주의: 같은 날짜가 여러 지표·분할에 반복될 수 있습니다.")
print("금융지표 변화 단위: 금리·스프레드는 bp, 환율은 %입니다.")
print("관측값은 삭제하지 않았습니다.")

In [ ]:
import matplotlib.pyplot as plt

# Windows 한국어 글꼴 설정
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

control_names = {
    "dual": "KOSPI·KOSDAQ",
    "kospi": "KOSPI"
}

factor_names = {
    "KR3Y_change": "국고채 3년",
    "KR10Y_change": "국고채 10년",
    "KR_CORP_AA3Y_change": "회사채 AA- 3년",
    "KR_CREDIT_change": "신용스프레드",
    "KR_CURVE_change": "장단기 금리차",
    "USDKRW_change": "달러/원"
}

# 1. 전체 156개 조합의 통합 MSE 개선율 분포
# 두 통제 방식에 동일한 구간과 가로축 적용
values = cv_pair_summary["pooled_mse_improvement_pct"].to_numpy()
assert np.isfinite(values).all()

bins = np.histogram_bin_edges(values, bins=25)

fig, axes = plt.subplots(
    1, 2, figsize=(12, 4.5), sharex=True, sharey=True
)

for ax, control in zip(axes, ["dual", "kospi"]):
    group = cv_pair_summary.loc[
        cv_pair_summary["control"] == control
    ]
    improvement = group["pooled_mse_improvement_pct"]

    ax.hist(
        improvement,
        bins=bins,
        color="#4477AA",
        edgecolor="white"
    )
    ax.axvline(0, color="black", linestyle="--", linewidth=1)
    ax.axvline(
        improvement.median(),
        color="#CC6677",
        linewidth=1.5,
        label=f"중앙값 {improvement.median():.3f}%"
    )

    ax.set_title(
        f"{control_names[control]}\n"
        f"개선 {(improvement > 0).sum()}/156개"
    )
    ax.set_xlabel("통합 MSE 개선율(%)")
    ax.legend()

axes[0].set_ylabel("업종·지표 조합 수")

fig.suptitle(
    "전체 일별 조합의 검증 오차 개선 분포\n"
    "양수: 금융지표 추가 시 개선 / 음수: 악화"
)
fig.tight_layout()

fig.savefig(
    OUT / "cv_improvement_distribution.png",
    dpi=180,
    bbox_inches="tight"
)
plt.show()

# 2. 세 기간 모두 개선된 조합의 분할별 개선율
# 결과를 보고 선택한 탐색적 부분집합임을 명시
chosen = cv_pair_summary.loc[
    cv_pair_summary["improved_folds"] == 3
].sort_values(
    ["control", "pooled_mse_improvement_pct"],
    ascending=[True, False]
).copy()

if not chosen.empty:
    columns = [
        "fold1_improvement_pct",
        "fold2_improvement_pct",
        "fold3_improvement_pct"
    ]

    matrix = chosen[columns].to_numpy()

    row_labels = [
        f"{row.sector_name} × {factor_names[row.factor]}"
        f" [{control_names[row.control]}]"
        for row in chosen.itertuples()
    ]

    fig, ax = plt.subplots(
        figsize=(10, max(5, len(chosen) * 0.43))
    )

    color_limit = max(float(np.abs(matrix).max()), 0.01)

    image = ax.imshow(
        matrix,
        cmap="RdBu",
        vmin=-color_limit,
        vmax=color_limit,
        aspect="auto"
    )

    ax.set_xticks([0, 1, 2], ["분할 1", "분할 2", "분할 3"])
    ax.set_yticks(range(len(chosen)), row_labels)

    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            value = matrix[i, j]
            text_color = (
                "white" if abs(value) > color_limit * 0.6
                else "black"
            )
            ax.text(
                j, i, f"{value:.3f}%",
                ha="center", va="center",
                color=text_color
            )

    fig.colorbar(image, ax=ax, label="MSE 개선율(%)")

    ax.set_title(
        "세 검증 기간 모두 개선된 조합\n"
        "결과 확인 후 선택한 탐색적 부분집합·통계적 유의성 판정 아님"
    )

    fig.tight_layout()
    fig.savefig(
        OUT / "cv_consistent_improvement_heatmap.png",
        dpi=180,
        bbox_inches="tight"
    )
    plt.show()

print("그림 저장 위치:", OUT)

## 일별 회귀분석 재현 및 시간 분할 검증 결과

### 1. 검증 목적과 방법

WI26 26개 업종과 국내 금융지표 6개의 관계를 대상으로,
기존 분석의 재현 여부와 이후 기간에서의 추가 설명력을 확인했다.

먼저 포함된 수준값으로 일별 수익률과 금융지표 변화량을
재계산하고, 기존 분석 결과와 비교했다.

이후 시간 순서를 유지한 확장형 3분할 검증을 수행했다.
각 분할의 과거 학습 구간에서 계수를 추정하고,
이후 검증 구간에는 계수를 고정 적용했다.

- 분할 1: 학습 362거래일, 검증 123거래일
- 분할 2: 학습 485거래일, 검증 120거래일
- 분할 3: 학습 605거래일, 검증 124거래일

KOSPI 단독 통제와 KOSPI·KOSDAQ 통제를 각각 적용했다.
시장지수만 사용하는 기준모형과 금융지표 하나를 추가한 모형을
동일한 학습·검증 행으로 비교했다.

개선율은 다음과 같이 계산했다.

`MSE 개선율(%) = 100 × (1 − 추가모형 MSE / 기준모형 MSE)`

양수는 오차 감소, 음수는 오차 증가를 의미한다.

### 2. 기존 결과의 재현

일별 입력 18,954행의 9개 수치 열을 수준값에서 재계산한 결과,
허용오차를 벗어난 불일치는 없었다.

기존 일별 결과와 2,652개 수치를 비교하고,
KOSPI 통제 계수·HAC 표준오차·p값·BY156 보정값 등
624개 수치를 추가 비교한 결과 모두 일치했다.

최초 KOSPI 통제 분석에서는 개별 p값이 0.05 미만인 조합이
3개였으나, BY156 보정 후에는 0개였다.

이는 비교한 기존 계산이 재현됐다는 의미이며,
관계의 유의성이나 예측 성능이 입증됐다는 뜻은 아니다.

### 3. 전체 조합의 검증 오차

| 항목 | KOSPI·KOSDAQ 통제 | KOSPI 통제 |
|---|---:|---:|
| 전체 조합 수 | 156 | 156 |
| 통합 MSE 개선 조합 수 | 41 | 35 |
| 세 기간 모두 개선된 조합 수 | 7 | 5 |
| 통합 개선율 중앙값 | -0.174% | -0.206% |
| 통합 개선율 평균 | -0.262% | -0.284% |

통합 MSE는 367개 검증일의 오차를 합산해 계산했다.
표의 평균과 중앙값은 각 조합의 통합 개선율을 요약한 값이다.

분포 그림에서는 두 통제 방식 모두 개선율이 0보다 작은
조합이 더 많고, 중앙값도 음수로 나타났다.
따라서 금융지표를 추가하면 전반적으로 설명력이 개선된다는
근거는 확인하지 못했다.

이는 해당 금융지표와 업종 사이에 어떠한 관계도 없다는
결론과는 구별된다.

### 4. 세 기간 모두 개선된 조합

두 시장 통제에서는 7개, KOSPI 통제에서는 5개 조합이
세 검증 기간 모두 MSE를 줄였다.

에너지 × 달러/원, 소프트웨어 × 장단기 금리차는
두 통제 방식에서 모두 세 기간의 MSE 개선이 관찰됐다.

다만 개선 폭은 작았고 기간별 차이도 있었다.
예를 들어 KOSPI 통제의 소프트웨어 × 장단기 금리차는
분할 2에서 약 1.004% 개선됐으나 분할 3에서는 약 0.146%였다.

열지도에는 세 기간 모두 개선된 조합만 표시했다.
따라서 전체 조합이 개선됐다는 인상을 주지 않도록
전체 분포 그림과 함께 해석해야 한다.

### 5. 계수와 오차 지표의 안정성

세 기간 모두 개선된 12개 모형 조합은 학습 계수가 모두
음수였으며, 세 학습 기간에서 부호가 유지됐다.

그러나 부호 유지가 계수 크기의 안정성이나 통계적 유의성을
보장하지는 않는다. 다수 조합은 각 학습 기간의 개별
95% 신뢰구간이 0을 포함했다.

또한 통신서비스 × 금리의 두 시장 통제 모형 등에서는
MSE가 개선됐지만 MAE는 악화했다.
큰 오차에 민감한 MSE 개선이 일반적인 절대오차 개선으로
이어지지 않을 수 있으므로 두 지표를 함께 보고했다.

### 6. 모형 진단

최대 VIF는 두 시장 통제에서 약 2.97,
KOSPI 통제에서 약 1.05였다.
표준화 조건수와 함께 볼 때 심한 공선성 징후는 두드러지지 않았다.

일부 학습 모형에는 잔차 자기상관·이분산의 참고표시가 있었다.
진단 p값은 다중검정 미보정 값이며 자동 탈락 기준으로 사용하지 않았다.

Cook 거리 점검에서는 2024-08-05, 2024-12-10,
2026-03-04 등이 영향 관측값으로 나타났다.
이 날짜들은 입력 출처와 관측값을 추가 확인할 대상으로 기록했으며,
관측값을 삭제하거나 윈저화하지 않았다.

### 7. 해석의 한계

- 검증일의 시장 수익률과 금융지표 변화를 입력으로 사용했다.
  따라서 실행 가능한 미래 예측이나 인과효과 검증이 아니다.
- 전체 기간 결과를 확인한 뒤 수행한 사후 기간 안정성 점검이다.
  아직 보지 않은 자료에 대한 최종 확증 시험은 아니다.
- 학습 기간이 겹치는 세 분할을 독립적인 세 실험으로 취급하지 않는다.
- 개선 조합은 많은 조합을 평가한 뒤 확인한 탐색적 결과이다.
  세 기간 모두 개선됐다는 사실만으로 통계적 유의성을 주장하지 않는다.
- WI26 최초 엑셀 추출, 공급기관 가격의 전기간 독립 대조,
  실제 발표시점은 검증하지 않았다.
- 5·10거래일 검증과 BY1560·BY2340 재현은 이번 일별 작업에 포함하지 않았다.

### 8. 결론

기존 일별 분석은 비교한 항목에서 재현됐다.
시간 분할 검증에서는 금융지표 추가에 따른 전반적이고
일관된 오차 개선은 확인하지 못했다.

일부 조합에서 작고 지속적인 개선이 관찰됐지만,
오차 지표 간 차이와 영향 관측값, 사후 탐색이라는 한계를 고려해
추가 확인이 필요한 결과로 정리한다.

## 5·10거래일 구간 독립 생성 및 기존 입력 대조

In [ ]:
# 1. 복리로 연결할 수익률과 합산할 bp 변화량 구분
return_columns = [
    "sector_return_pct",
    "kospi_return_pct",
    "kosdaq_return_pct",
    "USDKRW_change"
]

bp_columns = [
    "KR3Y_change",
    "KR10Y_change",
    "KR_CORP_AA3Y_change",
    "KR_CREDIT_change",
    "KR_CURVE_change"
]

value_columns = return_columns + bp_columns


def compound_percent(values):
    values = np.asarray(values, dtype=float)
    assert np.isfinite(values).all()
    return float(100 * (np.prod(1 + values / 100) - 1))


# 2. 일별 자료에서 비중첩 구간 직접 생성
blocks_by_horizon = {}
tail_rows = []
comparison_rows = []

all_dates = pd.DatetimeIndex(sorted(df["date"].unique()))

for horizon in [5, 10]:
    complete_blocks = len(all_dates) // horizon
    used_days = complete_blocks * horizon
    rows = []

    for code, group in df.groupby("sector_code", sort=True):
        group = group.sort_values("date").reset_index(drop=True)

        assert pd.DatetimeIndex(group["date"]).equals(all_dates)

        for block in range(complete_blocks):
            part = group.iloc[
                block * horizon:(block + 1) * horizon
            ]

            assert len(part) == horizon

            row = {
                "horizon_days": horizon,
                "block": block,
                "sector_code": code,
                "sector_name": part["sector_name"].iloc[0],
                "start_date": part["previous_sector_date"].iloc[0],
                "first_return_date": part["date"].iloc[0],
                "end_date": part["date"].iloc[-1]
            }

            for column in return_columns:
                row[column] = compound_percent(part[column])

            for column in bp_columns:
                row[column] = float(part[column].sum())

            rows.append(row)

    blocks = pd.DataFrame(rows)

    assert len(blocks) == complete_blocks * 26
    assert not blocks.duplicated(["sector_code", "block"]).any()

    # 인접 구간은 이전 종료 가격일과 다음 기준 가격일을 공유
    # 일별 수익률 자체는 중복하지 않음
    for code, group in blocks.groupby("sector_code"):
        group = group.sort_values("block")

        assert (
            group["end_date"].iloc[:-1].to_numpy()
            == group["start_date"].iloc[1:].to_numpy()
        ).all(), f"구간 연결 불일치: {code}"

    blocks_by_horizon[horizon] = blocks

    blocks.to_csv(
        OUT / f"recalculated_blocks_{horizon}d.csv",
        index=False,
        encoding="utf-8-sig"
    )

    # 마지막 불완전 구간에서 제외된 수익률 날짜 기록
    for day in all_dates[used_days:]:
        tail_rows.append({
            "horizon_days": horizon,
            "excluded_return_date": day,
            "reason": "마지막 불완전 구간"
        })

    # 3. 직접 생성한 뒤 팀원이 제공한 구간과 비교
    reference_blocks = pd.read_csv(
        ROOT / f"data/dual/blocks_{horizon}d_with_kosdaq.csv",
        parse_dates=["start_date", "first_return_date", "end_date"]
    )

    merged = blocks.merge(
        reference_blocks,
        on=["sector_code", "block"],
        suffixes=("_new", "_ref"),
        how="outer",
        validate="one_to_one",
        indicator=True
    )

    assert merged["_merge"].eq("both").all(), (
        f"{horizon}일 구간의 비교 대상이 다릅니다."
    )
    assert len(merged) == len(blocks)

    # 구간 식별 정보와 날짜는 정확히 일치해야 함
    exact_columns = [
        "horizon_days", "sector_name",
        "start_date", "first_return_date", "end_date"
    ]

    for column in exact_columns:
        matched = merged[f"{column}_new"].eq(
            merged[f"{column}_ref"]
        )

        comparison_rows.append({
            "horizon_days": horizon,
            "item": column,
            "checked_rows": len(merged),
            "mismatches": int((~matched).sum()),
            "max_absolute_error": np.nan
        })

    # 수치 열은 지정한 허용오차로 비교
    for column in value_columns:
        new = merged[f"{column}_new"].to_numpy(dtype=float)
        old = merged[f"{column}_ref"].to_numpy(dtype=float)

        finite = np.isfinite(new) & np.isfinite(old)
        matched = finite & np.isclose(
            new, old, atol=1e-8, rtol=1e-6
        )

        comparison_rows.append({
            "horizon_days": horizon,
            "item": column,
            "checked_rows": len(merged),
            "mismatches": int((~matched).sum()),
            "max_absolute_error": float(
                np.max(np.abs(new - old))
            )
        })

    print(
        f"{horizon}거래일 구간 생성: "
        f"업종별 {complete_blocks}개 / 전체 {len(blocks)}행"
    )
    print("마지막 종료일:", blocks["end_date"].max().date())
    print("불완전 구간 제외 거래일 수:", len(all_dates) - used_days)


# 4. 비교 결과와 제외 날짜 저장
block_audit = pd.DataFrame(comparison_rows)

excluded_tail = pd.DataFrame(
    tail_rows,
    columns=["horizon_days", "excluded_return_date", "reason"]
)

block_audit.to_csv(
    OUT / "block_reproduction_audit.csv",
    index=False,
    encoding="utf-8-sig"
)

excluded_tail.to_csv(
    OUT / "excluded_incomplete_blocks.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n【구간 입력 재현 대조】")
print(
    block_audit.rename(columns={
        "horizon_days": "구간길이",
        "item": "비교항목",
        "checked_rows": "비교행수",
        "mismatches": "불일치수",
        "max_absolute_error": "최대절대오차"
    }).to_string(index=False)
)

assert block_audit["mismatches"].sum() == 0, (
    "구간 생성 불일치를 먼저 확인해야 합니다."
)

print("\n통과: 직접 생성한 5·10거래일 구간이 팀원 입력과 일치합니다.")

In [ ]:
# 1. 구간 회귀 함수
def fit_block(target, predictors):
    design = sm.add_constant(predictors, has_constant="add")

    assert np.isfinite(target.to_numpy()).all()
    assert np.isfinite(design.to_numpy()).all()
    assert np.linalg.matrix_rank(design.to_numpy()) == design.shape[1]

    return sm.OLS(target, design, missing="raise").fit(
        cov_type="HAC",
        cov_kwds={
            "maxlags": 1,
            "kernel": "bartlett",
            "use_correction": True
        },
        use_t=False
    )


# 2. 동시점·시차에 맞춰 입력 정렬
block_results_rows = []
aligned_block_inputs = {}

for horizon, blocks in blocks_by_horizon.items():
    for mode in ["same_block", "lagged_next_block"]:
        aligned_parts = []

        for code, group in blocks.groupby("sector_code", sort=True):
            group = group.sort_values("block").reset_index(drop=True)

            if mode == "same_block":
                target_rows = group.copy().reset_index(drop=True)
                source_rows = group.copy().reset_index(drop=True)
            else:
                target_rows = group.iloc[1:].copy().reset_index(drop=True)
                source_rows = group.iloc[:-1].copy().reset_index(drop=True)

                assert (
                    source_rows["end_date"].to_numpy()
                    == target_rows["start_date"].to_numpy()
                ).all()

            aligned = target_rows[
                [
                    "sector_code", "sector_name", "block",
                    "start_date", "first_return_date", "end_date",
                    "sector_return_pct",
                    "kospi_return_pct", "kosdaq_return_pct"
                ]
            ].copy()

            aligned["x_start_date"] = source_rows["start_date"]
            aligned["x_end_date"] = source_rows["end_date"]

            for factor in factors:
                aligned[factor] = source_rows[factor].to_numpy()

            aligned_parts.append(aligned)

            target = aligned["sector_return_pct"]
            market_kospi = aligned[["kospi_return_pct"]]
            market_dual = aligned[
                ["kospi_return_pct", "kosdaq_return_pct"]
            ]

            baseline_kospi = fit_block(target, market_kospi)
            baseline_dual = fit_block(target, market_dual)

            for factor in factors:
                indicator = aligned[factor]

                total = fit_block(target, aligned[[factor]])
                expanded_kospi = fit_block(
                    target,
                    aligned[["kospi_return_pct", factor]]
                )
                expanded_dual = fit_block(
                    target,
                    aligned[
                        ["kospi_return_pct", "kosdaq_return_pct", factor]
                    ]
                )

                total_ci = total.conf_int().loc[factor]
                kospi_ci = expanded_kospi.conf_int().loc[factor]
                dual_ci = expanded_dual.conf_int().loc[factor]

                block_results_rows.append({
                    "horizon_days": horizon,
                    "mode": mode,
                    "sector_code": code,
                    "sector_name": aligned["sector_name"].iloc[0],
                    "factor": factor,
                    "n_obs": len(aligned),
                    "target_start": aligned[
                        "first_return_date"
                    ].min(),
                    "target_end": aligned["end_date"].max(),
                    "x_start": aligned["x_start_date"].min(),
                    "x_end": aligned["x_end_date"].max(),
                    "corr_total": target.corr(indicator),
                    "partial_corr_kospi_only": partial_corr(
                        target, indicator, market_kospi
                    ),
                    "partial_corr_dual": partial_corr(
                        target, indicator, market_dual
                    ),
                    "beta_total": total.params[factor],
                    "se_total_hac": total.bse[factor],
                    "p_total_hac": total.pvalues[factor],
                    "total_ci_low": total_ci.iloc[0],
                    "total_ci_high": total_ci.iloc[1],
                    "beta_kospi": expanded_kospi.params[factor],
                    "se_kospi_hac": expanded_kospi.bse[factor],
                    "p_kospi_only_hac": expanded_kospi.pvalues[factor],
                    "kospi_ci_low": kospi_ci.iloc[0],
                    "kospi_ci_high": kospi_ci.iloc[1],
                    "r2_kospi_baseline": baseline_kospi.rsquared,
                    "r2_kospi_expanded": expanded_kospi.rsquared,
                    "delta_adj_r2_kospi": (
                        expanded_kospi.rsquared_adj
                        - baseline_kospi.rsquared_adj
                    ),
                    "beta_dual": expanded_dual.params[factor],
                    "se_dual_hac": expanded_dual.bse[factor],
                    "p_dual_hac": expanded_dual.pvalues[factor],
                    "dual_ci_low": dual_ci.iloc[0],
                    "dual_ci_high": dual_ci.iloc[1],
                    "r2_total": total.rsquared,
                    "r2_dual_market_baseline": baseline_dual.rsquared,
                    "r2_dual_expanded": expanded_dual.rsquared,
                    "delta_adj_r2_dual": (
                        expanded_dual.rsquared_adj
                        - baseline_dual.rsquared_adj
                    ),
                    "hac_lags": 1
                })

        aligned_block_inputs[(horizon, mode)] = pd.concat(
            aligned_parts,
            ignore_index=True
        )

        print(f"{horizon}거래일 / {mode}: 계산 완료")


# 3. 결과와 정렬된 입력 저장
block_results = pd.DataFrame(block_results_rows)

assert len(block_results) == 624
assert not block_results.duplicated(
    ["horizon_days", "mode", "sector_code", "factor"]
).any()

block_results.to_csv(
    OUT / "block_recalculated_results.csv",
    index=False,
    encoding="utf-8-sig"
)

for (horizon, mode), table in aligned_block_inputs.items():
    table.to_csv(
        OUT / f"aligned_{horizon}d_{mode}.csv",
        index=False,
        encoding="utf-8-sig"
    )

print("\n【구간별 계산 확인】")
print(
    block_results.groupby(["horizon_days", "mode"])
    .agg(
        combinations=("factor", "size"),
        observations_min=("n_obs", "min"),
        observations_max=("n_obs", "max")
    )
    .reset_index()
    .rename(columns={
        "horizon_days": "구간길이",
        "mode": "분석방식",
        "combinations": "조합수",
        "observations_min": "최소표본수",
        "observations_max": "최대표본수"
    })
    .replace({
        "분석방식": {
            "same_block": "동시점",
            "lagged_next_block": "시차"
        }
    })
    .to_string(index=False)
)

print("\n전체 계산 조합 수:", len(block_results))

In [ ]:
# 1. 재계산한 일별·구간 결과 통합
daily_for_all = daily_results.copy()
daily_for_all["horizon_days"] = 1
daily_for_all["mode"] = "same_block"

all_results = pd.concat(
    [daily_for_all, block_results],
    ignore_index=True
)

result_keys = ["horizon_days", "mode", "sector_code", "factor"]

assert len(all_results) == 780
assert not all_results.duplicated(result_keys).any()

p_columns = ["p_total_hac", "p_dual_hac", "p_kospi_only_hac"]

assert np.isfinite(all_results[p_columns].to_numpy()).all()
assert (
    (all_results[p_columns] >= 0)
    & (all_results[p_columns] <= 1)
).all().all()


# 2. 함께 평가하는 전체 검정에 BY 보정 적용
# KOSPI 통제: 일별과 구간 전체 780개
all_results["p_kospi_by780"] = multipletests(
    all_results["p_kospi_only_hac"].to_numpy(),
    method="fdr_by"
)[1]

# 금융지표 단독 780개 + 두 시장 통제 780개
p1560 = np.concatenate([
    all_results["p_total_hac"].to_numpy(),
    all_results["p_dual_hac"].to_numpy()
])

assert len(p1560) == 1560

q1560 = multipletests(p1560, method="fdr_by")[1]

all_results["p_total_by1560"] = q1560[:780]
all_results["p_dual_by1560"] = q1560[780:]

# 위 1560개 + KOSPI 통제 780개
p2340 = np.concatenate([
    p1560,
    all_results["p_kospi_only_hac"].to_numpy()
])

assert len(p2340) == 2340

q2340 = multipletests(p2340, method="fdr_by")[1]

all_results["p_total_by2340"] = q2340[:780]
all_results["p_dual_by2340"] = q2340[780:1560]

# 5·10일 KOSPI 통제 624개만의 보정
block_with_by = all_results.loc[
    all_results["horizon_days"].isin([5, 10])
].copy()

assert len(block_with_by) == 624

block_with_by["p_kospi_by624"] = multipletests(
    block_with_by["p_kospi_only_hac"].to_numpy(),
    method="fdr_by"
)[1]


# 3. 기존 결과를 읽어 재계산 결과와 비교
dual_reference = pd.read_csv(
    ROOT / "reference/dual/model_comparison.csv"
)

horizon_reference = pd.read_csv(
    ROOT / "reference/horizon/horizon_regression_results.csv"
)

comparison_records = []


def compare_numeric_tables(new_table, old_table, mapping, label):
    assert not new_table.duplicated(result_keys).any()
    assert not old_table.duplicated(result_keys).any()

    new_columns = list(mapping.keys())
    old_columns = list(dict.fromkeys(mapping.values()))

    left = new_table[result_keys + new_columns].rename(
        columns={column: f"new__{column}" for column in new_columns}
    )
    right = old_table[result_keys + old_columns].rename(
        columns={column: f"old__{column}" for column in old_columns}
    )

    paired = left.merge(
        right,
        on=result_keys,
        how="outer",
        validate="one_to_one",
        indicator=True
    )

    assert paired["_merge"].eq("both").all(), (
        f"{label}: 비교 대상 조합이 다릅니다."
    )

    for _, row in paired.iterrows():
        for new_column, old_column in mapping.items():
            actual = float(row[f"new__{new_column}"])
            expected = float(row[f"old__{old_column}"])

            finite = np.isfinite(actual) and np.isfinite(expected)
            matched = finite and (
                actual == expected if new_column == "n_obs"
                else np.isclose(
                    actual, expected, atol=1e-8, rtol=1e-6
                )
            )

            comparison_records.append({
                **{key: row[key] for key in result_keys},
                "comparison_group": label,
                "item": new_column,
                "reference_value": expected,
                "recalculated_value": actual,
                "absolute_error": abs(actual - expected),
                "relative_error": (
                    abs(actual - expected) / abs(expected)
                    if expected != 0 else np.nan
                ),
                "matched": bool(matched)
            })


# 종합 결과: 모든 구간 780개 조합과 BY1560·BY2340 대조
dual_columns = [
    "n_obs",
    "corr_total",
    "partial_corr_kospi_only",
    "partial_corr_dual",
    "beta_total",
    "p_total_hac",
    "total_ci_low",
    "total_ci_high",
    "beta_dual",
    "p_dual_hac",
    "dual_ci_low",
    "dual_ci_high",
    "p_kospi_only_hac",
    "r2_total",
    "r2_dual_market_baseline",
    "r2_dual_expanded",
    "delta_adj_r2_dual",
    "p_total_by1560",
    "p_dual_by1560",
    "p_total_by2340",
    "p_dual_by2340"
]

compare_numeric_tables(
    all_results,
    dual_reference,
    {column: column for column in dual_columns},
    "전체 구간 종합 결과"
)

# 기존 KOSPI 구간 결과: 계수·표준오차·보정 등을 별도 대조
horizon_mapping = {
    "n_obs": "n_obs",
    "corr_total": "corr_raw",
    "partial_corr_kospi_only": "partial_corr",
    "beta_kospi": "beta_factor",
    "se_kospi_hac": "se_hac",
    "p_kospi_only_hac": "p_hac",
    "kospi_ci_low": "ci_low",
    "kospi_ci_high": "ci_high",
    "r2_kospi_baseline": "r2_baseline",
    "r2_kospi_expanded": "r2_expanded",
    "delta_adj_r2_kospi": "delta_adj_r2",
    "p_kospi_by624": "p_by_new624",
    "p_kospi_by780": "p_by_all780"
}

compare_numeric_tables(
    block_with_by,
    horizon_reference,
    horizon_mapping,
    "KOSPI 구간 결과"
)


# 4. 결과 저장 및 요약
full_reproduction_diff = pd.DataFrame(comparison_records)

all_results.to_csv(
    OUT / "all_horizons_recalculated_results.csv",
    index=False,
    encoding="utf-8-sig"
)

block_with_by.to_csv(
    OUT / "block_results_with_by.csv",
    index=False,
    encoding="utf-8-sig"
)

full_reproduction_diff.to_csv(
    OUT / "all_horizons_reproduction_diff.csv",
    index=False,
    encoding="utf-8-sig"
)

print("【기존 결과와의 전체 대조】")
print("비교 항목 수:", len(full_reproduction_diff))
print(
    "불일치 수:",
    int((~full_reproduction_diff["matched"]).sum())
)
print(
    "최대 절대오차:",
    full_reproduction_diff["absolute_error"].max()
)

print("\n【BY 보정 후 0.05 미만인 검정 수】")
for column in [
    "p_kospi_by780",
    "p_total_by1560", "p_dual_by1560",
    "p_total_by2340", "p_dual_by2340"
]:
    print(column, ":", int((all_results[column] < 0.05).sum()))

print(
    "p_kospi_by624:",
    int((block_with_by["p_kospi_by624"] < 0.05).sum())
)

if full_reproduction_diff["matched"].all():
    print("\n통과: 비교한 회귀 결과와 BY 보정값이 모두 일치합니다.")
else:
    print("\n불일치 항목:")
    print(
        full_reproduction_diff.loc[
            ~full_reproduction_diff["matched"]
        ].head(20).to_string(index=False)
    )
    raise ValueError("불일치 원인을 먼저 확인해야 합니다.")

In [ ]:
# 1. 고정된 5·10거래일 입력을 시간 분할에 배정
block_cv_splits = {}
block_split_rows = []
boundary_exclusions = []

for (horizon, mode), panel in aligned_block_inputs.items():
    panel = panel.copy()

    date_columns = [
        "start_date", "first_return_date", "end_date",
        "x_start_date", "x_end_date"
    ]

    for column in date_columns:
        panel[column] = pd.to_datetime(panel[column])

    for plan in split_plan.itertuples(index=False):
        selected_periods = {}

        for period, period_start, period_end in [
            ("train", plan.train_start, plan.train_end),
            ("test", plan.test_start, plan.test_end)
        ]:
            # 목표 수익률을 구성하는 모든 거래일이 기간 안에 있어야 함
            inside = (
                (panel["first_return_date"] >= period_start)
                & (panel["end_date"] <= period_end)
            )

            # 기간에 일부 걸치지만 완전히 포함되지 않는 구간 식별
            overlaps = (
                (panel["first_return_date"] <= period_end)
                & (panel["end_date"] >= period_start)
            )
            crosses_boundary = overlaps & ~inside

            selected = (
                panel.loc[inside]
                .sort_values(["sector_code", "end_date"])
                .copy()
            )

            excluded = panel.loc[crosses_boundary].copy()

            assert not selected.empty
            assert selected["first_return_date"].ge(period_start).all()
            assert selected["end_date"].le(period_end).all()

            counts = selected.groupby("sector_code").size()

            assert len(counts) == 26
            assert counts.nunique() == 1

            # 모든 업종이 같은 목표 구간을 사용하는지 확인
            expected_dates = None

            for code, group in selected.groupby("sector_code"):
                endpoints = tuple(group["end_date"])

                if expected_dates is None:
                    expected_dates = endpoints
                else:
                    assert endpoints == expected_dates, (
                        f"목표 구간 불일치: {code}"
                    )

            selected_periods[period] = selected

            # 날짜 기준 제외 구간 수와 업종별 행 수를 구분
            excluded_blocks = excluded[
                ["first_return_date", "end_date"]
            ].drop_duplicates()

            block_split_rows.append({
                "horizon_days": horizon,
                "mode": mode,
                "fold": plan.fold,
                "period": period,
                "planned_start": period_start,
                "planned_end": period_end,
                "actual_first_return": selected[
                    "first_return_date"
                ].min(),
                "actual_last_return": selected["end_date"].max(),
                "blocks_per_sector": int(counts.iloc[0]),
                "sector_count": len(counts),
                "total_rows": len(selected),
                "boundary_blocks_excluded": len(excluded_blocks)
            })

            if not excluded.empty:
                excluded["horizon_days"] = horizon
                excluded["mode"] = mode
                excluded["fold"] = plan.fold
                excluded["period"] = period
                excluded["reason"] = "목표 수익률 구간이 분할 경계를 가로지름"

                boundary_exclusions.append(
                    excluded[
                        [
                            "horizon_days", "mode", "fold", "period",
                            "sector_code", "block",
                            "first_return_date", "end_date", "reason"
                        ]
                    ]
                )

        train = selected_periods["train"]
        test = selected_periods["test"]

        # 학습 목표가 끝난 뒤 검증 목표가 시작해야 함
        assert train["end_date"].max() < test[
            "first_return_date"
        ].min()

        assert set(train["end_date"]).isdisjoint(
            set(test["end_date"])
        )

        # 시차 입력이 목표 구간의 기준가격일에 연결되는지 확인
        if mode == "lagged_next_block":
            for selected in [train, test]:
                assert (
                    selected["x_end_date"]
                    == selected["start_date"]
                ).all()

        block_cv_splits[(horizon, mode, plan.fold)] = {
            "train": train,
            "test": test
        }


# 2. 실제 분할과 제외 사유 저장
block_splits = pd.DataFrame(block_split_rows)

exclusion_columns = [
    "horizon_days", "mode", "fold", "period",
    "sector_code", "block",
    "first_return_date", "end_date", "reason"
]

if boundary_exclusions:
    excluded_boundaries = pd.concat(
        boundary_exclusions,
        ignore_index=True
    )
else:
    excluded_boundaries = pd.DataFrame(columns=exclusion_columns)

assert len(block_cv_splits) == 12
assert len(block_splits) == 24

block_splits.to_csv(
    OUT / "block_splits.csv",
    index=False,
    encoding="utf-8-sig"
)

excluded_boundaries.to_csv(
    OUT / "excluded_boundary_blocks.csv",
    index=False,
    encoding="utf-8-sig"
)


# 3. 검증 기간의 실제 표본 수 확인
test_summary = block_splits.loc[
    block_splits["period"] == "test",
    [
        "horizon_days", "mode", "fold",
        "actual_first_return", "actual_last_return",
        "blocks_per_sector", "boundary_blocks_excluded"
    ]
].copy()

test_summary["mode"] = test_summary["mode"].map({
    "same_block": "동시점",
    "lagged_next_block": "시차"
})

print("【5·10거래일 검증 구간 확인】")
print(
    test_summary.rename(columns={
        "horizon_days": "구간길이",
        "mode": "분석방식",
        "fold": "분할",
        "actual_first_return": "최초수익률일",
        "actual_last_return": "최종수익률일",
        "blocks_per_sector": "업종별검증구간수",
        "boundary_blocks_excluded": "경계제외구간수"
    }).to_string(index=False)
)

print("\n통과: 원래 구간 시작점을 유지했습니다.")
print("통과: 학습·검증 경계를 가로지르는 목표 구간을 제외했습니다.")
print("통과: 학습 목표와 검증 목표가 시간적으로 겹치지 않습니다.")
print("\n주의: 제외 기록은 분할·기간별 기록이므로 같은 구간이 반복될 수 있습니다.")

In [ ]:
# 1. 구간 교차검증 결과를 담을 목록
block_metric_rows = []
block_fitted_frames = []
block_coefficient_rows = []

for (horizon, mode, fold), split in block_cv_splits.items():
    train_panel = split["train"]
    test_panel = split["test"]

    for code, train_group in train_panel.groupby(
        "sector_code", sort=True
    ):
        train = train_group.sort_values("end_date").copy()
        test = (
            test_panel.loc[test_panel["sector_code"] == code]
            .sort_values("end_date")
            .copy()
        )

        assert not test.empty
        assert train["end_date"].max() < test[
            "first_return_date"
        ].min()

        for control, market_columns in control_sets.items():
            # 기준모형은 학습 구간에서만 추정
            baseline = fit_block(
                train["sector_return_pct"],
                train[market_columns]
            )

            baseline_design = sm.add_constant(
                test[market_columns],
                has_constant="add"
            ).reindex(columns=baseline.model.exog_names)

            baseline_pred = np.asarray(
                baseline.predict(baseline_design),
                dtype=float
            )

            actual = test["sector_return_pct"].to_numpy(dtype=float)

            assert np.isfinite(actual).all()
            assert np.isfinite(baseline_pred).all()

            baseline_errors = error_metrics(actual, baseline_pred)

            for factor in factors:
                predictors = market_columns + [factor]

                # 추가모형도 기준모형과 동일한 학습 행 사용
                expanded = fit_block(
                    train["sector_return_pct"],
                    train[predictors]
                )

                assert baseline.nobs == expanded.nobs == len(train)

                expanded_design = sm.add_constant(
                    test[predictors],
                    has_constant="add"
                ).reindex(columns=expanded.model.exog_names)

                expanded_pred = np.asarray(
                    expanded.predict(expanded_design),
                    dtype=float
                )

                assert np.isfinite(expanded_pred).all()

                expanded_errors = error_metrics(actual, expanded_pred)

                improvement = (
                    100 * (
                        1 - expanded_errors["mse"]
                        / baseline_errors["mse"]
                    )
                    if baseline_errors["mse"] > 0 else np.nan
                )

                # 구간 길이와 분석 방식까지 포함한 고유 식별자
                model_id = (
                    f"{code}__{factor}__{control}"
                    f"__{horizon}d__{mode}"
                )

                block_metric_rows.append({
                    "model_id": model_id,
                    "horizon_days": horizon,
                    "mode": mode,
                    "fold": fold,
                    "sector_code": code,
                    "sector_name": train["sector_name"].iloc[0],
                    "factor": factor,
                    "control": control,
                    "n_train": len(train),
                    "n_test": len(test),
                    "train_start": train["first_return_date"].min(),
                    "train_end": train["end_date"].max(),
                    "test_start": test["first_return_date"].min(),
                    "test_end": test["end_date"].max(),
                    "mse_baseline": baseline_errors["mse"],
                    "mse_expanded": expanded_errors["mse"],
                    "rmse_baseline": baseline_errors["rmse"],
                    "rmse_expanded": expanded_errors["rmse"],
                    "mae_baseline": baseline_errors["mae"],
                    "mae_expanded": expanded_errors["mae"],
                    "mse_improvement_pct": improvement
                })

                # 검증 구간별 실측값과 계산값 저장
                block_fitted_frames.append(pd.DataFrame({
                    "model_id": model_id,
                    "horizon_days": horizon,
                    "mode": mode,
                    "fold": fold,
                    "sector_code": code,
                    "factor": factor,
                    "control": control,
                    "first_return_date": (
                        test["first_return_date"].to_numpy()
                    ),
                    "date": test["end_date"].to_numpy(),
                    "x_start_date": test["x_start_date"].to_numpy(),
                    "x_end_date": test["x_end_date"].to_numpy(),
                    "actual": actual,
                    "pred_baseline": baseline_pred,
                    "pred_expanded": expanded_pred,
                    "error_baseline": actual - baseline_pred,
                    "error_expanded": actual - expanded_pred
                }))

                # 학습 구간의 금융지표 계수 기록
                ci = expanded.conf_int().loc[factor]

                block_coefficient_rows.append({
                    "model_id": model_id,
                    "horizon_days": horizon,
                    "mode": mode,
                    "fold": fold,
                    "sector_code": code,
                    "factor": factor,
                    "control": control,
                    "n_train": len(train),
                    "train_start": train["first_return_date"].min(),
                    "train_end": train["end_date"].max(),
                    "beta_factor": expanded.params[factor],
                    "se_hac": expanded.bse[factor],
                    "ci_low": ci.iloc[0],
                    "ci_high": ci.iloc[1],
                    "p_hac": expanded.pvalues[factor],
                    "p_adjustment": "미보정·기술 목적",
                    "hac_maxlags": 1
                })

    mode_label = "동시점" if mode == "same_block" else "시차"
    print(f"{horizon}거래일 / {mode_label} / 분할 {fold} 완료")


# 2. 결과 구성 및 중복 검사
block_cv_metrics = pd.DataFrame(block_metric_rows)
block_cv_fitted = pd.concat(
    block_fitted_frames,
    ignore_index=True
)
block_coefficient_stability = pd.DataFrame(block_coefficient_rows)

assert len(block_cv_metrics) == 3744
assert len(block_coefficient_stability) == 3744

assert not block_cv_metrics.duplicated(
    ["model_id", "fold"]
).any()

assert not block_cv_fitted.duplicated(
    ["model_id", "fold", "date"]
).any()

assert block_cv_metrics.groupby("model_id").size().eq(3).all()


# 3. 원자료 수준의 평가 결과 저장
for filename, table in {
    "block_cv_metrics.csv": block_cv_metrics,
    "block_cv_fitted_values.csv": block_cv_fitted,
    "block_coefficient_stability.csv": block_coefficient_stability
}.items():
    table.to_csv(
        OUT / filename,
        index=False,
        encoding="utf-8-sig"
    )


# 4. 구간·방식·통제·분할별 결과 요약
block_fold_summary = (
    block_cv_metrics.groupby(
        ["horizon_days", "mode", "control", "fold"]
    )
    .agg(
        evaluated_pairs=("model_id", "size"),
        n_test=("n_test", "first"),
        improved_pairs=(
            "mse_improvement_pct",
            lambda values: int((values > 0).sum())
        ),
        median_improvement_pct=("mse_improvement_pct", "median"),
        mean_improvement_pct=("mse_improvement_pct", "mean"),
        worst_improvement_pct=("mse_improvement_pct", "min")
    )
    .reset_index()
)

block_fold_summary.to_csv(
    OUT / "block_cv_fold_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

display_summary = block_fold_summary.rename(columns={
    "horizon_days": "구간길이",
    "mode": "분석방식",
    "control": "통제방식",
    "fold": "분할",
    "evaluated_pairs": "평가조합수",
    "n_test": "검증구간수",
    "improved_pairs": "개선조합수",
    "median_improvement_pct": "개선율중앙값(%)",
    "mean_improvement_pct": "개선율평균(%)",
    "worst_improvement_pct": "최저개선율(%)"
}).replace({
    "분석방식": {
        "same_block": "동시점",
        "lagged_next_block": "시차"
    },
    "통제방식": {
        "kospi": "KOSPI",
        "dual": "KOSPI·KOSDAQ"
    }
})

print("\n【5·10거래일 시간 분할 검증 결과】")
print(display_summary.to_string(index=False))

print("\n전체 평가 수:", len(block_cv_metrics))
print("저장 위치:", OUT)
print("주의: 학습 계수의 p값은 미보정 참고값이며 후보 선택에 사용하지 않습니다.")

In [ ]:
# 1. 업종·지표·통제 방식별로 세 분할의 검증 성적 통합
group_keys = [
    "model_id", "horizon_days", "mode",
    "sector_code", "sector_name", "factor", "control"
]

block_pair_rows = []

for key, group in block_cv_metrics.groupby(group_keys, sort=True):
    assert len(group) == 3
    assert set(group["fold"]) == {1, 2, 3}

    n_total = int(group["n_test"].sum())

    # 분할별 표본 수로 가중하여 전체 검증 구간의 오차 계산
    mse_baseline = float(
        (group["mse_baseline"] * group["n_test"]).sum() / n_total
    )
    mse_expanded = float(
        (group["mse_expanded"] * group["n_test"]).sum() / n_total
    )
    mae_baseline = float(
        (group["mae_baseline"] * group["n_test"]).sum() / n_total
    )
    mae_expanded = float(
        (group["mae_expanded"] * group["n_test"]).sum() / n_total
    )

    row = dict(zip(group_keys, key))

    row.update({
        "n_test_total": n_total,
        "improved_folds": int(
            (group["mse_improvement_pct"] > 0).sum()
        ),
        "worst_fold_improvement_pct": float(
            group["mse_improvement_pct"].min()
        ),
        "pooled_mse_baseline": mse_baseline,
        "pooled_mse_expanded": mse_expanded,
        "pooled_rmse_baseline": float(np.sqrt(mse_baseline)),
        "pooled_rmse_expanded": float(np.sqrt(mse_expanded)),
        "pooled_mae_baseline": mae_baseline,
        "pooled_mae_expanded": mae_expanded,
        "pooled_mse_improvement_pct": (
            100 * (1 - mse_expanded / mse_baseline)
            if mse_baseline > 0 else np.nan
        ),
        "mae_improvement_pct": (
            100 * (1 - mae_expanded / mae_baseline)
            if mae_baseline > 0 else np.nan
        )
    })

    for fold in [1, 2, 3]:
        row[f"fold{fold}_improvement_pct"] = float(
            group.loc[
                group["fold"] == fold, "mse_improvement_pct"
            ].iloc[0]
        )

    block_pair_rows.append(row)

block_pair_summary = pd.DataFrame(block_pair_rows)

assert len(block_pair_summary) == 1248
assert not block_pair_summary.duplicated("model_id").any()


# 2. 학습 기간별 금융지표 계수의 방향과 신뢰구간 확인
coefficient_rows_summary = []

for model_id, group in block_coefficient_stability.groupby(
    "model_id", sort=True
):
    group = group.sort_values("fold")

    assert len(group) == 3
    assert set(group["fold"]) == {1, 2, 3}

    numeric_columns = ["beta_factor", "ci_low", "ci_high"]
    assert np.isfinite(group[numeric_columns].to_numpy()).all()

    beta = group["beta_factor"].to_numpy()

    row = {
        "model_id": model_id,
        "same_nonzero_sign": bool(
            (beta > 0).all() or (beta < 0).all()
        ),
        "beta_range": float(beta.max() - beta.min()),
        "ci_includes_zero_folds": int((
            (group["ci_low"] <= 0)
            & (group["ci_high"] >= 0)
        ).sum())
    }

    for fold in [1, 2, 3]:
        row[f"beta_fold{fold}"] = float(
            group.loc[
                group["fold"] == fold, "beta_factor"
            ].iloc[0]
        )

    coefficient_rows_summary.append(row)

block_coefficient_summary = pd.DataFrame(
    coefficient_rows_summary
)

assert len(block_coefficient_summary) == 1248

block_pair_summary = block_pair_summary.merge(
    block_coefficient_summary,
    on="model_id",
    how="left",
    validate="one_to_one"
)

assert block_pair_summary["same_nonzero_sign"].notna().all()


# 3. 검증 구간별 오차에서 통합 MSE를 직접 재계산
direct_rows = []

for model_id, group in block_cv_fitted.groupby("model_id"):
    direct_rows.append({
        "model_id": model_id,
        "direct_n_test": len(group),
        "direct_mse_baseline": float(
            np.mean(group["error_baseline"].to_numpy() ** 2)
        ),
        "direct_mse_expanded": float(
            np.mean(group["error_expanded"].to_numpy() ** 2)
        )
    })

direct_check = block_pair_summary.merge(
    pd.DataFrame(direct_rows),
    on="model_id",
    how="outer",
    validate="one_to_one",
    indicator=True
)

assert direct_check["_merge"].eq("both").all()

assert (
    direct_check["n_test_total"]
    == direct_check["direct_n_test"]
).all()

assert np.allclose(
    direct_check["pooled_mse_baseline"],
    direct_check["direct_mse_baseline"],
    atol=1e-8,
    rtol=1e-6
)

assert np.allclose(
    direct_check["pooled_mse_expanded"],
    direct_check["direct_mse_expanded"],
    atol=1e-8,
    rtol=1e-6
)


# 4. 구간 길이·분석 방식·시장 통제별 전체 결과 요약
overview_rows = []

for key, group in block_pair_summary.groupby(
    ["horizon_days", "mode", "control"], sort=True
):
    horizon, mode, control = key

    assert len(group) == 156
    assert group["n_test_total"].nunique() == 1

    overview_rows.append({
        "horizon_days": horizon,
        "mode": mode,
        "control": control,
        "total_pairs": len(group),
        "n_test_total": int(group["n_test_total"].iloc[0]),
        "pooled_improved_pairs": int(
            (group["pooled_mse_improvement_pct"] > 0).sum()
        ),
        "all_3_folds_improved": int(
            (group["improved_folds"] == 3).sum()
        ),
        "all_3_improved_same_sign": int((
            (group["improved_folds"] == 3)
            & group["same_nonzero_sign"]
        ).sum()),
        "median_improvement_pct": float(
            group["pooled_mse_improvement_pct"].median()
        ),
        "mean_improvement_pct": float(
            group["pooled_mse_improvement_pct"].mean()
        )
    })

block_overview = pd.DataFrame(overview_rows)

assert len(block_overview) == 8


# 5. 유리한 결과와 불리한 결과를 모두 저장
block_pair_summary.to_csv(
    OUT / "block_cv_pair_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

block_overview.to_csv(
    OUT / "block_cv_overall_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

direct_check.drop(columns="_merge").to_csv(
    OUT / "block_cv_pooled_error_check.csv",
    index=False,
    encoding="utf-8-sig"
)


# 6. 화면 출력용 열 이름과 분석 방식 표시
display_table = block_overview.rename(columns={
    "horizon_days": "구간길이",
    "mode": "분석방식",
    "control": "통제방식",
    "total_pairs": "전체조합수",
    "n_test_total": "통합검증구간수",
    "pooled_improved_pairs": "통합개선조합수",
    "all_3_folds_improved": "3개기간모두개선",
    "all_3_improved_same_sign": "모두개선및부호유지",
    "median_improvement_pct": "개선율중앙값(%)",
    "mean_improvement_pct": "개선율평균(%)"
}).replace({
    "분석방식": {
        "same_block": "동시점",
        "lagged_next_block": "시차"
    },
    "통제방식": {
        "kospi": "KOSPI",
        "dual": "KOSPI·KOSDAQ"
    }
})

print("통과: 검증 표본 수와 직접 재계산한 통합 MSE가 일치합니다.")
print("\n【5·10거래일 통합 검증 결과】")
print(display_table.to_string(index=False))

print("\n저장한 파일:")
print("- block_cv_pair_summary.csv")
print("- block_cv_overall_summary.csv")
print("- block_cv_pooled_error_check.csv")
print("\n저장 위치:", OUT)

In [ ]:
# 1. 구간 모형의 진단 결과를 담을 목록
block_diagnostic_rows = []
block_vif_rows = []
block_influence_rows = []

for (horizon, mode, fold), split in block_cv_splits.items():
    train_panel = split["train"]

    for code, group in train_panel.groupby("sector_code", sort=True):
        train = group.sort_values("end_date").copy()

        for control, market_columns in control_sets.items():
            for factor in factors:
                model_id = (
                    f"{code}__{factor}__{control}"
                    f"__{horizon}d__{mode}"
                )

                predictors = train[market_columns + [factor]]
                design = sm.add_constant(
                    predictors,
                    has_constant="add"
                )

                assert np.isfinite(design.to_numpy()).all()
                assert (
                    np.linalg.matrix_rank(design.to_numpy())
                    == design.shape[1]
                )

                # 진단용 일반 OLS 적합
                # HAC 적용 여부와 관계없이 계수와 잔차는 동일
                model = sm.OLS(
                    train["sector_return_pct"],
                    design,
                    missing="raise"
                ).fit()

                residuals = model.resid.to_numpy()

                # 구간 단위 1시차의 잔차 자기상관 점검
                lb = acorr_ljungbox(
                    residuals,
                    lags=[1],
                    return_df=True
                )

                # 정규성 가정을 완화한 이분산 진단
                bp_lm, bp_p, _, _ = het_breuschpagan(
                    residuals,
                    design,
                    robust=True
                )

                influence = model.get_influence()
                cooks = influence.cooks_distance[0]
                leverage = influence.hat_matrix_diag

                # 절편을 제외한 모든 설명변수의 VIF
                vif_values = []

                for position, variable in enumerate(design.columns):
                    if variable == "const":
                        continue

                    value = float(variance_inflation_factor(
                        design.to_numpy(), position
                    ))
                    vif_values.append(value)

                    block_vif_rows.append({
                        "model_id": model_id,
                        "horizon_days": horizon,
                        "mode": mode,
                        "fold": fold,
                        "variable": variable,
                        "vif": value
                    })

                # 단위 차이를 줄인 표준화 조건수
                scales = predictors.std(ddof=0)
                assert (scales > 0).all()

                standardized = (
                    predictors - predictors.mean()
                ) / scales

                scaled_design = sm.add_constant(
                    standardized,
                    has_constant="add"
                )

                block_diagnostic_rows.append({
                    "model_id": model_id,
                    "horizon_days": horizon,
                    "mode": mode,
                    "fold": fold,
                    "sector_code": code,
                    "factor": factor,
                    "control": control,
                    "n_train": len(train),
                    "rank": np.linalg.matrix_rank(design.to_numpy()),
                    "n_parameters": design.shape[1],
                    "condition_number_raw": np.linalg.cond(
                        design.to_numpy()
                    ),
                    "condition_number_scaled": np.linalg.cond(
                        scaled_design.to_numpy()
                    ),
                    "max_vif": max(vif_values),
                    "ljung_box_lag1_p": float(
                        lb["lb_pvalue"].iloc[0]
                    ),
                    "koenker_lm": float(bp_lm),
                    "koenker_p": float(bp_p),
                    "max_cooks_distance": float(cooks.max()),
                    "max_leverage": float(leverage.max())
                })

                # 영향이 큰 학습 구간 3개를 기록하며 삭제하지 않음
                for position in np.argsort(cooks)[-3:][::-1]:
                    block_influence_rows.append({
                        "model_id": model_id,
                        "horizon_days": horizon,
                        "mode": mode,
                        "fold": fold,
                        "sector_code": code,
                        "factor": factor,
                        "control": control,
                        "first_return_date": train[
                            "first_return_date"
                        ].iloc[position],
                        "end_date": train["end_date"].iloc[position],
                        "cooks_distance": float(cooks[position]),
                        "leverage": float(leverage[position]),
                        "residual": float(residuals[position])
                    })

    mode_label = "동시점" if mode == "same_block" else "시차"
    print(f"{horizon}거래일 / {mode_label} / 분할 {fold} 진단 완료")


# 2. 전체 결과 저장
block_model_diagnostics = pd.DataFrame(block_diagnostic_rows)
block_full_model_vif = pd.DataFrame(block_vif_rows)
block_influential_dates = pd.DataFrame(block_influence_rows)

assert len(block_model_diagnostics) == 3744
assert not block_model_diagnostics.duplicated(
    ["model_id", "fold"]
).any()

for filename, table in {
    "block_model_diagnostics.csv": block_model_diagnostics,
    "block_full_model_vif.csv": block_full_model_vif,
    "block_influential_training_intervals.csv": block_influential_dates
}.items():
    table.to_csv(
        OUT / filename,
        index=False,
        encoding="utf-8-sig"
    )


# 3. 구간 길이·분석 방식·통제 방식별 요약
block_diagnostic_summary = (
    block_model_diagnostics.groupby(
        ["horizon_days", "mode", "control"]
    )
    .agg(
        model_count=("model_id", "size"),
        max_vif=("max_vif", "max"),
        max_scaled_condition=("condition_number_scaled", "max"),
        autocorrelation_flags=(
            "ljung_box_lag1_p",
            lambda values: int((values < 0.05).sum())
        ),
        heteroskedasticity_flags=(
            "koenker_p",
            lambda values: int((values < 0.05).sum())
        ),
        max_cooks_distance=("max_cooks_distance", "max")
    )
    .reset_index()
)

block_diagnostic_summary.to_csv(
    OUT / "block_diagnostic_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

display_table = block_diagnostic_summary.rename(columns={
    "horizon_days": "구간길이",
    "mode": "분석방식",
    "control": "통제방식",
    "model_count": "모형수",
    "max_vif": "최대VIF",
    "max_scaled_condition": "최대표준화조건수",
    "autocorrelation_flags": "자기상관참고표시수",
    "heteroskedasticity_flags": "이분산참고표시수",
    "max_cooks_distance": "최대Cook거리"
}).replace({
    "분석방식": {
        "same_block": "동시점",
        "lagged_next_block": "시차"
    },
    "통제방식": {
        "kospi": "KOSPI",
        "dual": "KOSPI·KOSDAQ"
    }
})

print("\n【5·10거래일 학습 모형 진단】")
print(display_table.to_string(index=False))
print("\n주의: 진단 p값은 미보정 참고값이며 자동 채택 기준이 아닙니다.")
print("자기상관은 구간 단위 1시차를 검사하여 일별 5시차 진단과 구별합니다.")
print("관측값을 삭제하거나 검증 결과에 맞춰 모형을 변경하지 않았습니다.")

In [ ]:
# 1. 제출용 결과 폴더 생성
SUBMIT = OUT.parent / "submission"
SUBMIT.mkdir(exist_ok=True)


# 2. 일별 결과에 구간 식별 정보 추가
daily_metrics_submit = cv_metrics.copy()
daily_metrics_submit["horizon_days"] = 1
daily_metrics_submit["mode"] = "same_block"

daily_coefficients_submit = coefficient_stability.copy()
daily_coefficients_submit["horizon_days"] = 1
daily_coefficients_submit["mode"] = "same_block"

daily_fitted_submit = cv_fitted.copy()
daily_fitted_submit["horizon_days"] = 1
daily_fitted_submit["mode"] = "same_block"

# 일별 목표 수익률의 시작일은 해당 거래일
daily_fitted_submit["first_return_date"] = daily_fitted_submit["date"]


# 3. 일별과 5·10거래일 결과 통합
all_cv_metrics = pd.concat(
    [daily_metrics_submit, block_cv_metrics],
    ignore_index=True,
    sort=False
)

all_cv_coefficients = pd.concat(
    [daily_coefficients_submit, block_coefficient_stability],
    ignore_index=True,
    sort=False
)

all_cv_fitted = pd.concat(
    [daily_fitted_submit, block_cv_fitted],
    ignore_index=True,
    sort=False
)

daily_pair_submit = cv_pair_summary.copy()
daily_pair_submit["horizon_days"] = 1
daily_pair_submit["mode"] = "same_block"

# 일별 조합 요약에 고유 식별자 연결
daily_ids = cv_metrics[
    ["sector_code", "factor", "control", "model_id"]
].drop_duplicates()

daily_pair_submit = daily_pair_submit.merge(
    daily_ids,
    on=["sector_code", "factor", "control"],
    validate="one_to_one"
)

all_cv_pair_summary = pd.concat(
    [daily_pair_submit, block_pair_summary],
    ignore_index=True,
    sort=False
)


# 4. 전체 결과의 개수와 중복 확인
assert len(all_cv_metrics) == 4680
assert len(all_cv_coefficients) == 4680
assert len(all_cv_pair_summary) == 1560

assert not all_cv_metrics.duplicated(
    ["model_id", "fold"]
).any()

assert not all_cv_coefficients.duplicated(
    ["model_id", "fold"]
).any()

assert not all_cv_pair_summary.duplicated("model_id").any()

assert not all_cv_fitted.duplicated(
    ["model_id", "fold", "date"]
).any()

assert all_cv_metrics.groupby("model_id").size().eq(3).all()

# 保存内容に欠損がないか主要数値を確認
metric_columns = [
    "mse_baseline", "mse_expanded",
    "rmse_baseline", "rmse_expanded",
    "mae_baseline", "mae_expanded"
]

assert np.isfinite(
    all_cv_metrics[metric_columns].to_numpy()
).all()


# 5. 学習係数に学習期間情報を補完
training_info = all_cv_metrics[
    ["model_id", "fold", "train_start", "train_end"]
]

all_cv_coefficients = all_cv_coefficients.drop(
    columns=["train_start", "train_end"],
    errors="ignore"
).merge(
    training_info,
    on=["model_id", "fold"],
    how="left",
    validate="one_to_one"
)

assert all_cv_coefficients[
    ["train_start", "train_end"]
].notna().all().all()


# 6. 全条件の概要表を作成
all_overview_rows = []

for key, group in all_cv_pair_summary.groupby(
    ["horizon_days", "mode", "control"],
    sort=True
):
    horizon, mode, control = key

    assert len(group) == 156

    all_overview_rows.append({
        "horizon_days": horizon,
        "mode": mode,
        "control": control,
        "total_pairs": len(group),
        "n_test_total": int(group["n_test_total"].iloc[0]),
        "pooled_improved_pairs": int(
            (group["pooled_mse_improvement_pct"] > 0).sum()
        ),
        "all_3_folds_improved": int(
            (group["improved_folds"] == 3).sum()
        ),
        "median_improvement_pct": float(
            group["pooled_mse_improvement_pct"].median()
        ),
        "mean_improvement_pct": float(
            group["pooled_mse_improvement_pct"].mean()
        )
    })

all_cv_overview = pd.DataFrame(all_overview_rows)

assert len(all_cv_overview) == 10


# 7. 統合した結果を保存
tables = {
    "cv_metrics.csv": all_cv_metrics,
    "cv_fitted_values.csv": all_cv_fitted,
    "coefficient_stability.csv": all_cv_coefficients,
    "cv_pair_summary.csv": all_cv_pair_summary,
    "cv_overall_summary.csv": all_cv_overview
}

for filename, table in tables.items():
    table.to_csv(
        SUBMIT / filename,
        index=False,
        encoding="utf-8-sig"
    )

print("【전체 결과 통합 확인】")
print("분할별 평가 수:", len(all_cv_metrics))
print("학습 계수 기록 수:", len(all_cv_coefficients))
print("전체 모형 조합 수:", len(all_cv_pair_summary))
print("검증 구간별 계산값 수:", len(all_cv_fitted))

print("\n【일별·5일·10일 전체 요약】")
print(
    all_cv_overview.rename(columns={
        "horizon_days": "구간길이",
        "mode": "분석방식",
        "control": "통제방식",
        "total_pairs": "전체조합수",
        "n_test_total": "통합검증관측수",
        "pooled_improved_pairs": "통합개선조합수",
        "all_3_folds_improved": "3개기간모두개선",
        "median_improvement_pct": "개선율중앙값(%)",
        "mean_improvement_pct": "개선율평균(%)"
    })
    .replace({
        "분석방식": {
            "same_block": "동시점",
            "lagged_next_block": "시차"
        },
        "통제방식": {
            "kospi": "KOSPI",
            "dual": "KOSPI·KOSDAQ"
        }
    })
    .to_string(index=False)
)

print("\n제출용 결과 저장 위치:", SUBMIT)

In [ ]:
# 1. 결과와 무관하게 최초 확인 조합을 진단 대상으로 고정
PLOT_CODE = "WI100"
PLOT_FACTOR = "KR3Y_change"
PLOT_CONTROL = "dual"

plot_markets = ["kospi_return_pct", "kosdaq_return_pct"]

FIGURES = SUBMIT / "figures"
FIGURES.mkdir(exist_ok=True)

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

scenarios = [
    (1, "same_block"),
    (5, "same_block"),
    (5, "lagged_next_block"),
    (10, "same_block"),
    (10, "lagged_next_block")
]


def make_design(table, columns):
    return sm.add_constant(
        table[columns],
        has_constant="add"
    )


for horizon, mode in scenarios:
    fig, axes = plt.subplots(3, 3, figsize=(15, 12))

    for row_number, fold in enumerate([1, 2, 3]):
        if horizon == 1:
            sector = df.loc[
                df["sector_code"] == PLOT_CODE
            ].sort_values("date")

            train = sector.loc[
                sector["date"].isin(
                    cv_splits[fold]["train_dates"]
                )
            ].copy()

            test = sector.loc[
                sector["date"].isin(
                    cv_splits[fold]["test_dates"]
                )
            ].copy()

            fit_function = fit_daily

        else:
            selected_split = block_cv_splits[
                (horizon, mode, fold)
            ]

            train = selected_split["train"].loc[
                selected_split["train"]["sector_code"] == PLOT_CODE
            ].sort_values("end_date").copy()

            test = selected_split["test"].loc[
                selected_split["test"]["sector_code"] == PLOT_CODE
            ].sort_values("end_date").copy()

            fit_function = fit_block

        train_market_design = make_design(train, plot_markets)
        test_market_design = make_design(test, plot_markets)

        # 학습 자료로만 업종과 금융지표의 시장 통제식 추정
        market_y_model = sm.OLS(
            train["sector_return_pct"],
            train_market_design,
            missing="raise"
        ).fit()

        market_x_model = sm.OLS(
            train[PLOT_FACTOR],
            train_market_design,
            missing="raise"
        ).fit()

        train_y_residual = market_y_model.resid.to_numpy()
        train_x_residual = market_x_model.resid.to_numpy()

        # 검증 자료에는 학습한 시장 통제 계수를 고정 적용
        test_y_residual = (
            test["sector_return_pct"].to_numpy()
            - np.asarray(
                market_y_model.predict(test_market_design)
            )
        )

        test_x_residual = (
            test[PLOT_FACTOR].to_numpy()
            - np.asarray(
                market_x_model.predict(test_market_design)
            )
        )

        # 금융지표 추가모형도 학습 구간에서만 추정
        predictors = plot_markets + [PLOT_FACTOR]

        expanded = fit_function(
            train["sector_return_pct"],
            train[predictors]
        )

        train_pred = np.asarray(
            expanded.predict(make_design(train, predictors))
        )
        test_pred = np.asarray(
            expanded.predict(make_design(test, predictors))
        )

        train_error = (
            train["sector_return_pct"].to_numpy() - train_pred
        )
        test_error = (
            test["sector_return_pct"].to_numpy() - test_pred
        )

        # 두 산점도의 축 범위를 같게 설정
        x_values = np.concatenate([
            train_x_residual, test_x_residual
        ])
        y_values = np.concatenate([
            train_y_residual, test_y_residual
        ])

        x_min, x_max = float(x_values.min()), float(x_values.max())
        y_min, y_max = float(y_values.min()), float(y_values.max())

        x_margin = max((x_max - x_min) * 0.05, 0.01)
        y_margin = max((y_max - y_min) * 0.05, 0.01)

        for column_number, x_residual, y_residual, label, color in [
            (
                0, train_x_residual, train_y_residual,
                "학습", "#4477AA"
            ),
            (
                1, test_x_residual, test_y_residual,
                "검증", "#CC6677"
            )
        ]:
            ax = axes[row_number, column_number]

            ax.scatter(
                x_residual, y_residual,
                s=15, alpha=0.55, color=color
            )

            ax.axhline(0, color="gray", linewidth=0.6)
            ax.axvline(0, color="gray", linewidth=0.6)

            ax.set_xlim(x_min - x_margin, x_max + x_margin)
            ax.set_ylim(y_min - y_margin, y_max + y_margin)

            ax.set_title(
                f"분할 {fold} · {label} 산점도 "
                f"· 관측치 {len(x_residual)}개"
            )
            ax.set_xlabel("시장 통제 후 국고채 3년 변화 잔차(bp)")
            ax.set_ylabel("시장 통제 후 업종 수익률 잔차(%p)")

        # 추가모형의 적합값과 잔차 비교
        ax = axes[row_number, 2]

        ax.scatter(
            train_pred, train_error,
            s=15, alpha=0.35,
            color="#4477AA", label="학습"
        )
        ax.scatter(
            test_pred, test_error,
            s=22, alpha=0.7,
            color="#CC6677", label="검증"
        )

        ax.axhline(0, color="black", linestyle="--", linewidth=0.8)
        ax.set_title(f"분할 {fold} · 추가모형 잔차")
        ax.set_xlabel("모형 계산 수익률(%)")
        ax.set_ylabel("실측값 − 계산값(%p)")
        ax.legend()

    mode_label = "동시점" if mode == "same_block" else "시차"

    fig.suptitle(
        f"에너지 × 국고채 3년 · 두 시장 통제 · "
        f"{horizon}거래일 {mode_label}\n"
        "검증 기간에는 학습 계수를 고정 적용",
        fontsize=14
    )

    fig.tight_layout(rect=[0, 0, 1, 0.94])

    filename = f"diagnostic_{horizon}d_{mode}.png"

    fig.savefig(
        FIGURES / filename,
        dpi=160,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)

    print(f"저장 완료: {filename}")

print("\n그림 저장 위치:", FIGURES)

### 산점도 및 잔차 점검

결과의 유불리와 관계없이 최초 재현에 사용한 에너지 × 국고채 3년
조합을 고정하여 일별·5일·10일의 학습·검증 산점도와 잔차를 확인했다.

시장 통제식과 금융지표 추가모형은 학습 구간에서만 추정했으며,
검증 구간에는 학습한 계수를 고정 적용했다.

산점도에서는 기간을 넘어 일관되게 유지되는 강한 직선 관계가
뚜렷하지 않았다. 검증 구간에도 큰 잔차가 남았으며,
10거래일 분석은 분할당 검증 관측치가 11개뿐이어서
소수 관측값의 영향을 주의해야 한다.

구간 길이가 다르면 수익률 단위의 변동 폭도 달라지므로,
잔차의 절대 크기만으로 일별·5일·10일 모형의 우열을 판단하지 않는다.
모형의 추가 기여는 같은 조건의 기준모형과 비교한 검증 오차로 평가한다.

이 그림은 한 조합의 예시 진단이며 전체 조합에 대한 결론을 대신하지 않는다.
관측값은 삭제하지 않았다.

In [ ]:
import sys
import json
import hashlib
import platform
import importlib.metadata
from datetime import datetime, timezone, timedelta

# 1. 실제 실행 환경 기록
package_names = [
    "numpy", "pandas", "scipy", "statsmodels", "matplotlib"
]

package_versions = {
    name: importlib.metadata.version(name)
    for name in package_names
}

korea_timezone = timezone(timedelta(hours=9))

environment = {
    "recorded_at": datetime.now(korea_timezone).isoformat(),
    "python_version": platform.python_version(),
    "python_executable": sys.executable,
    "platform": platform.platform(),
    "packages": package_versions
}

(SUBMIT / "environment.json").write_text(
    json.dumps(environment, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

(SUBMIT / "requirements.txt").write_text(
    "\n".join(
        f"{name}=={version}"
        for name, version in package_versions.items()
    ) + "\n",
    encoding="utf-8"
)


# 2. 팀원 배포 명세의 모든 파일 해시 대조
original_manifest = pd.read_csv(
    ROOT / "FILE_MANIFEST.csv",
    dtype={"path": str, "sha256": str}
)

hash_rows = []
root_resolved = ROOT.resolve()

for entry in original_manifest.itertuples(index=False):
    relative_path = str(entry.path).replace("\\", "/")
    path = (ROOT / relative_path).resolve()

    # 전달자료 폴더 내부 경로인지 확인
    assert path.is_relative_to(root_resolved), (
        f"전달자료 외부 경로: {relative_path}"
    )

    exists = path.is_file()
    actual_hash = (
        hashlib.sha256(path.read_bytes()).hexdigest()
        if exists else ""
    )
    expected_hash = str(entry.sha256).lower()

    hash_rows.append({
        "relative_path": relative_path,
        "exists": exists,
        "expected_sha256": expected_hash,
        "actual_sha256": actual_hash,
        "hash_matched": exists and actual_hash == expected_hash,
        "expected_bytes": int(entry.bytes),
        "actual_bytes": path.stat().st_size if exists else None,
        "size_matched": (
            exists and path.stat().st_size == int(entry.bytes)
        )
    })

hash_audit = pd.DataFrame(hash_rows)

hash_audit.to_csv(
    SUBMIT / "handoff_hash_audit.csv",
    index=False,
    encoding="utf-8-sig"
)


# 3. 분석·재현 검증에 사용한 주요 입력의 지문 기록
input_roles = {
    "data/wi26/wi26_daily_history.csv":
        "업종 지수 수준값, 수익률 재계산",
    "data/financial/daily_financial_history.csv":
        "금리·환율·스프레드 수준값, 변화량 재계산",
    "data/domestic/kospi_daily_history.csv":
        "KOSPI 수준값, 시장 수익률 재계산",
    "data/dual/kosdaq_daily_history.csv":
        "KOSDAQ 수준값, 시장 수익률 재계산",
    "data/dual/daily_inputs_with_kosdaq.csv":
        "일별 분석 입력 및 비중첩 구간 생성",
    "data/dual/blocks_5d_with_kosdaq.csv":
        "독립 생성한 5거래일 입력과 비교",
    "data/dual/blocks_10d_with_kosdaq.csv":
        "독립 생성한 10거래일 입력과 비교",
    "reference/initial/regression_comparison.csv":
        "최초 일별 KOSPI 통제 결과와 비교",
    "reference/horizon/horizon_regression_results.csv":
        "5·10거래일 KOSPI 통제 결과와 비교",
    "reference/dual/model_comparison.csv":
        "종합 결과 및 BY1560·BY2340과 비교",
    "docs/VALIDATION_REQUEST.md":
        "검증 조건과 제출 요구사항 확인"
}

input_rows = []

for relative_path, role in input_roles.items():
    path = ROOT / relative_path
    assert path.is_file(), f"파일 없음: {relative_path}"

    input_rows.append({
        "relative_path": relative_path,
        "role": role,
        "bytes": path.stat().st_size,
        "sha256": hashlib.sha256(path.read_bytes()).hexdigest()
    })

input_manifest = pd.DataFrame(input_rows)

input_manifest.to_csv(
    SUBMIT / "input_manifest.csv",
    index=False,
    encoding="utf-8-sig"
)


# 4. 결과 출력
print("【실행 환경】")
print("Python:", environment["python_version"])

for name, version in package_versions.items():
    print(f"{name}: {version}")

print("\n【팀원 배포 파일 검사】")
print("명세 파일 수:", len(hash_audit))
print("없는 파일 수:", int((~hash_audit["exists"]).sum()))
print("해시 불일치 수:", int((~hash_audit["hash_matched"]).sum()))
print("크기 불일치 수:", int((~hash_audit["size_matched"]).sum()))

print("\n주요 입력 기록 수:", len(input_manifest))
print("저장 위치:", SUBMIT)

if not (
    hash_audit["hash_matched"].all()
    and hash_audit["size_matched"].all()
):
    print("\n【확인이 필요한 파일】")
    print(
        hash_audit.loc[
            ~hash_audit["hash_matched"]
            | ~hash_audit["size_matched"]
        ].to_string(index=False)
    )
else:
    print("\n통과: 배포 명세와 현재 파일의 해시·크기가 일치합니다.")

print("주의: 해시 일치는 공급기관 데이터의 정확성을 보증하지 않습니다.")

In [ ]:
# 1. 금융지표 수준값에서 스프레드 정의 검증
financial_levels = pd.read_csv(
    ROOT / "data/financial/daily_financial_history.csv"
)

assert not financial_levels.duplicated(
    ["indicator_id", "observation_period"]
).any()

levels_wide = financial_levels.pivot(
    index="observation_period",
    columns="indicator_id",
    values="value"
)

spread_definitions = {
    "KR_CREDIT": (
        "100 × (회사채 AA- 3년 − 국고채 3년)",
        "KR_CORP_AA3Y",
        "KR3Y"
    ),
    "KR_CURVE": (
        "100 × (국고채 10년 − 국고채 3년)",
        "KR10Y",
        "KR3Y"
    )
}

spread_rows = []

for indicator, definition in spread_definitions.items():
    formula, first, second = definition
    columns = [indicator, first, second]

    # 세 수준값이 모두 존재하는 날짜만 비교
    selected = levels_wide[columns].dropna()
    assert not selected.empty
    assert np.isfinite(selected.to_numpy()).all()

    recalculated = 100 * (selected[first] - selected[second])
    supplied = selected[indicator]

    matched = np.isclose(
        recalculated,
        supplied,
        atol=1e-8,
        rtol=1e-6
    )

    spread_rows.append({
        "indicator": indicator,
        "formula": formula,
        "unit": "bp",
        "checked_rows": len(selected),
        "mismatches": int((~matched).sum()),
        "max_absolute_error": float(
            np.max(np.abs(recalculated - supplied))
        )
    })

spread_audit = pd.DataFrame(spread_rows)

spread_audit.to_csv(
    SUBMIT / "spread_definition_audit.csv",
    index=False,
    encoding="utf-8-sig"
)

assert spread_audit["mismatches"].sum() == 0, (
    "스프레드 정의 불일치를 먼저 확인해야 합니다."
)


# 2. 주요 입력의 출처·기간·단위 보완
# 출처는 전달자료의 설명과 메타데이터 기준이며 외부 독립 검증과 구별
metadata = {
    "data/wi26/wi26_daily_history.csv": {
        "source": "팀원 제공 WI26 지수 CSV. 최초 엑셀은 미포함",
        "period": "2023-11-06~2026-11-05",
        "units": "업종 지수 수준값"
    },
    "data/financial/daily_financial_history.csv": {
        "source": "전달자료 기록상 ECOS·FRED 및 계산 스프레드",
        "period": "요청 기간 2023-11-06~2026-11-05. 실제 기간은 지표별로 다름",
        "units": "금리 연%, 스프레드 bp, 환율 원/달러, 유가 달러/배럴·달러/갤런"
    },
    "data/domestic/kospi_daily_history.csv": {
        "source": "전달자료 기록상 네이버 금융 차트",
        "period": "2023-11-06~2026-11-05",
        "units": "KOSPI 지수 수준값"
    },
    "data/dual/kosdaq_daily_history.csv": {
        "source": "전달자료 기록상 네이버 금융 차트",
        "period": "2023-11-06~2026-11-05",
        "units": "KOSDAQ 지수 수준값"
    },
    "data/dual/daily_inputs_with_kosdaq.csv": {
        "source": "팀원이 수준값으로 생성한 일별 입력. 이번 작업에서 변환 재검산",
        "period": "2023-11-07~2026-11-05",
        "units": "업종·시장·환율 변화율 %, 금리·스프레드 변화 bp"
    },
    "data/dual/blocks_5d_with_kosdaq.csv": {
        "source": "팀원 제공 5거래일 입력. 독립 생성 구간과 대조",
        "period": "기준가격일 2023-11-06, 마지막 종료일 2026-10-30",
        "units": "업종·시장·환율 누적 변화율 %, 금리·스프레드 누적 변화 bp"
    },
    "data/dual/blocks_10d_with_kosdaq.csv": {
        "source": "팀원 제공 10거래일 입력. 독립 생성 구간과 대조",
        "period": "기준가격일 2023-11-06, 마지막 종료일 2026-10-23",
        "units": "업종·시장·환율 누적 변화율 %, 금리·스프레드 누적 변화 bp"
    },
    "reference/initial/regression_comparison.csv": {
        "source": "팀원의 최초 KOSPI 통제 일별 회귀 결과",
        "period": "2023-11-07~2026-11-05",
        "units": "상관·설명력·p값은 무차원, 계수는 금융지표 단위당 수익률 %p"
    },
    "reference/horizon/horizon_regression_results.csv": {
        "source": "팀원의 5·10거래일 KOSPI 통제 회귀 결과",
        "period": "구간·분석 방식별 목표 기간은 파일의 날짜 열 참조",
        "units": "상관·설명력·p값은 무차원, 계수는 금융지표 단위당 수익률 %p"
    },
    "reference/dual/model_comparison.csv": {
        "source": "팀원의 금융지표 단독 및 두 시장 통제 종합 결과",
        "period": "일별·5일·10일 및 동시점·시차별 기간은 파일 참조",
        "units": "상관·설명력·p값은 무차원, 계수는 금융지표 단위당 수익률 %p"
    },
    "docs/VALIDATION_REQUEST.md": {
        "source": "팀원 제공 검증 요청서",
        "period": "작성 기준 2026-11-06",
        "units": "해당 없음"
    }
}

# 앞서 저장한 입력 명세를 읽어 보완
input_manifest = pd.read_csv(SUBMIT / "input_manifest.csv")

input_manifest["relative_path"] = (
    input_manifest["relative_path"]
    .str.replace("\\", "/", regex=False)
)

for field in ["source", "period", "units"]:
    input_manifest[field] = input_manifest["relative_path"].map(
        lambda path: metadata.get(path, {}).get(field)
    )

assert input_manifest[
    ["source", "period", "units"]
].notna().all().all(), "출처·기간·단위가 누락된 파일이 있습니다."

input_manifest.to_csv(
    SUBMIT / "input_manifest.csv",
    index=False,
    encoding="utf-8-sig"
)


# 3. 금융지표별 실제 확보 기간 기록
financial_coverage = (
    financial_levels.groupby(
        ["indicator_id", "indicator_name", "provider", "unit"],
        dropna=False
    )
    .agg(
        actual_start=("observation_period", "min"),
        actual_end=("observation_period", "max"),
        row_count=("observation_period", "size"),
        missing_values=("value", lambda values: int(values.isna().sum()))
    )
    .reset_index()
)

financial_coverage.to_csv(
    SUBMIT / "financial_source_coverage.csv",
    index=False,
    encoding="utf-8-sig"
)


# 4. 검증 범위와 미검증 사항 기록
scope_notes = """# 자료 출처와 검증 범위

- 출처 표기는 전달자료의 설명서와 메타데이터를 기준으로 작성했다.
- 배포 명세 60개 파일의 존재·SHA-256·크기가 모두 일치했다.
- 포함된 수준값에서 일별 분석 입력의 변환을 재검산했다.
- 일별 자료에서 비중첩 5·10거래일 구간을 독립 생성해 기존 입력과 비교했다.
- 신용스프레드는 100 × (회사채 AA- 3년 − 국고채 3년)으로 확인했다.
- 장단기 금리차는 100 × (국고채 10년 − 국고채 3년)으로 확인했다.
- 금리 수준값의 단위는 연%, 스프레드 수준값의 단위는 bp이다.
- 해외 지표는 기본 회귀에 사용하지 않았다.
- WI26 최초 엑셀부터 CSV까지의 추출 과정은 원본 미포함으로 검증하지 못했다.
- 공급기관 가격의 전기간 독립 대조와 실제 발표시점은 검증하지 않았다.
- 파일 해시 일치는 공급기관 자료 자체의 정확성을 보증하지 않는다.
"""

(SUBMIT / "data_scope_notes.md").write_text(
    scope_notes,
    encoding="utf-8"
)


# 5. 결과 출력
print("【스프레드 정의 검증】")
print(
    spread_audit.rename(columns={
        "indicator": "지표",
        "formula": "계산식",
        "unit": "단위",
        "checked_rows": "검사행수",
        "mismatches": "불일치수",
        "max_absolute_error": "최대절대오차"
    }).to_string(index=False)
)

print("\n출처·기간·단위를 기록한 입력 수:", len(input_manifest))

print("\n저장한 파일:")
print("- spread_definition_audit.csv")
print("- input_manifest.csv")
print("- financial_source_coverage.csv")
print("- data_scope_notes.md")
print("\n저장 위치:", SUBMIT)

In [ ]:
# 1. 감사표 작성에 사용할 기존 검사 결과 읽기
transformation_check = pd.read_csv(
    OUT / "transformation_audit.csv"
)
block_input_check = pd.read_csv(
    OUT / "block_reproduction_audit.csv"
)
spread_check = pd.read_csv(
    SUBMIT / "spread_definition_audit.csv"
)

audit_records = []


def add_audit(item, scope, result, detail):
    audit_records.append({
        "item": item,
        "scope": scope,
        "result": result,
        "detail": detail
    })


# 2. 일별 입력 구조와 계산 검사 기록
add_audit(
    "업종·날짜 중복",
    "일별 입력",
    "일치" if not df.duplicated(["sector_code", "date"]).any()
    else "불일치",
    f"전체 {len(df)}행, 업종 {df['sector_code'].nunique()}개"
)

numeric_columns = [
    "sector_return_pct", "kospi_return_pct",
    "kosdaq_return_pct", *factors
]

nonfinite_count = int(
    (~np.isfinite(df[numeric_columns].to_numpy())).sum()
)

add_audit(
    "결측·무한대",
    "일별 분석 수치",
    "일치" if nonfinite_count == 0 else "불일치",
    f"유한하지 않은 값 {nonfinite_count}개"
)

for row in transformation_check.itertuples(index=False):
    add_audit(
        row.column,
        "수준값에서 일별 변화량 재계산",
        "일치" if row.mismatches == 0 else "불일치",
        f"검사 {row.checked_rows}행, 최대절대오차 {row.max_abs_error}"
    )

for row in block_input_check.itertuples(index=False):
    add_audit(
        row.item,
        f"{row.horizon_days}거래일 구간 입력",
        "일치" if row.mismatches == 0 else "불일치",
        f"검사 {row.checked_rows}행"
    )

for row in spread_check.itertuples(index=False):
    add_audit(
        row.indicator,
        "스프레드 차감 방향과 수준값",
        "일치" if row.mismatches == 0 else "불일치",
        f"{row.formula}, 검사 {row.checked_rows}행"
    )

add_audit(
    "구간 제외",
    "마지막 불완전 구간",
    "기록 완료",
    "5일은 4거래일, 10일은 9거래일 제외. 제외 날짜는 별도 파일 참조"
)

add_audit(
    "구간 제외",
    "시간 분할 경계",
    "기록 완료",
    "목표 수익률 구간이 경계를 가로지르면 제외. 분할별 제외 기록 참조"
)

for item, reason in [
    ("최초 엑셀 추출", "WI26 최초 엑셀이 전달자료에 없어 검증하지 못함"),
    ("외부 가격 대조", "공급기관 가격의 전기간 독립 대조 미수행"),
    ("실제 발표시점", "과거 정보 취득 가능 시각 미검증"),
    ("공식 거래일 완전성", "공식 거래소 달력과의 독립 전수 대조 미수행")
]:
    add_audit(item, "검증 범위의 한계", "미검증", reason)

data_audit = pd.DataFrame(audit_records)

data_audit.to_csv(
    SUBMIT / "data_audit.csv",
    index=False,
    encoding="utf-8-sig"
)


# 3. 대표 날짜 선정: 결과가 유리한 날짜를 선택하지 않음
example_sector = (
    df.loc[df["sector_code"] == "WI100"]
    .sort_values("date")
    .copy()
)

example_sector["calendar_gap"] = (
    example_sector["date"] - example_sector["previous_sector_date"]
).dt.days

ordinary = example_sector.loc[
    example_sector["calendar_gap"] == 1
].iloc[0]

after_holiday = example_sector.loc[
    example_sector["calendar_gap"].idxmax()
]

large_move = example_sector.loc[
    example_sector["sector_return_pct"].abs().idxmax()
]

examples = [
    ("일반 거래일", ordinary),
    ("관측 간 달력 간격 최대일", after_holiday),
    ("해당 업종 절대수익률 최대일", large_move)
]


# 4. 원래 수준값으로 계산 과정을 기록
wi_levels = pd.read_csv(
    ROOT / "data/wi26/wi26_daily_history.csv"
).set_index(["sector_code", "date"])["index_value"]

financial_map = financial_levels.set_index(
    ["indicator_id", "observation_period"]
)["value"]

market_maps = {
    "kospi_return_pct": pd.read_csv(
        ROOT / "data/domestic/kospi_daily_history.csv"
    ).set_index("date")["kospi_index"],
    "kosdaq_return_pct": pd.read_csv(
        ROOT / "data/dual/kosdaq_daily_history.csv"
    ).set_index("date")["kosdaq_index"]
}

example_rows = []

for selection, row in examples:
    start = row["previous_sector_date"].strftime("%Y-%m-%d")
    end = row["date"].strftime("%Y-%m-%d")

    for column in numeric_columns:
        if column == "sector_return_pct":
            before = float(wi_levels.loc[("WI100", start)])
            after = float(wi_levels.loc[("WI100", end)])
            formula = "100 × (종료값 / 시작값 − 1)"
            calculated = 100 * (after / before - 1)
            unit = "%"

        elif column in market_maps:
            before = float(market_maps[column].loc[start])
            after = float(market_maps[column].loc[end])
            formula = "100 × (종료값 / 시작값 − 1)"
            calculated = 100 * (after / before - 1)
            unit = "%"

        else:
            indicator = column.removesuffix("_change")
            before = float(financial_map.loc[(indicator, start)])
            after = float(financial_map.loc[(indicator, end)])

            if indicator == "USDKRW":
                formula = "100 × (종료값 / 시작값 − 1)"
                calculated = 100 * (after / before - 1)
                unit = "%"
            elif indicator in ["KR_CREDIT", "KR_CURVE"]:
                formula = "종료값 − 시작값"
                calculated = after - before
                unit = "bp"
            else:
                formula = "100 × (종료값 − 시작값)"
                calculated = 100 * (after - before)
                unit = "bp"

        supplied = float(row[column])

        example_rows.append({
            "selection": selection,
            "sector_code": "WI100",
            "start_date": start,
            "end_date": end,
            "calendar_gap": int(row["calendar_gap"]),
            "item": column,
            "start_value": before,
            "end_value": after,
            "formula": formula,
            "calculated_change": calculated,
            "supplied_change": supplied,
            "change_unit": unit,
            "matched": bool(np.isclose(
                calculated, supplied, atol=1e-8, rtol=1e-6
            )),
            "human_review": "미확인"
        })

calculation_examples = pd.DataFrame(example_rows)
assert calculation_examples["matched"].all()

calculation_examples.to_csv(
    SUBMIT / "calculation_examples.csv",
    index=False,
    encoding="utf-8-sig"
)

print("【자료 감사 기록】")
print(data_audit.groupby("result").size().to_string())

print("\n【대표 날짜】")
print(
    calculation_examples[
        ["selection", "start_date", "end_date", "calendar_gap"]
    ].drop_duplicates().to_string(index=False)
)

print("\n계산 예시 수:", len(calculation_examples))
print("계산 예시 불일치 수:", int(
    (~calculation_examples["matched"]).sum()
))
print("\n저장 완료: data_audit.csv")
print("저장 완료: calculation_examples.csv")
print("사람의 손계산 확인은 아직 미확인으로 기록했습니다.")

In [ ]:
# 최종 보고서 작성
from pathlib import Path
import shutil


# 1. 추가 패키지 없이 마크다운 표 작성
def markdown_table(table):
    table = table.copy().fillna("")

    def format_value(value):
        if isinstance(value, (float, np.floating)):
            return f"{value:.6f}"
        return str(value).replace("|", "/").replace("\n", " ")

    header = "| " + " | ".join(map(str, table.columns)) + " |"
    separator = "| " + " | ".join(["---"] * len(table.columns)) + " |"

    rows = [
        "| " + " | ".join(format_value(value) for value in row) + " |"
        for row in table.itertuples(index=False, name=None)
    ]

    return "\n".join([header, separator, *rows])


# 2. 보고서에 사용할 전체 검증 성적
report_overview = all_cv_overview.rename(columns={
    "horizon_days": "구간길이",
    "mode": "분석방식",
    "control": "통제방식",
    "total_pairs": "전체조합수",
    "n_test_total": "통합검증관측수",
    "pooled_improved_pairs": "통합개선조합수",
    "all_3_folds_improved": "세기간모두개선",
    "median_improvement_pct": "개선율중앙값(%)",
    "mean_improvement_pct": "개선율평균(%)"
}).replace({
    "분석방식": {
        "same_block": "동시점",
        "lagged_next_block": "시차"
    },
    "통제방식": {
        "kospi": "KOSPI",
        "dual": "KOSPI·KOSDAQ"
    }
})

audit_status = (
    data_audit.groupby("result")
    .size()
    .rename_axis("상태")
    .reset_index(name="기록수")
)

# 손계산 상태는 파일의 실제 기록을 읽어 반영
saved_examples = pd.read_csv(
    SUBMIT / "calculation_examples.csv"
)

human_checked = int(
    saved_examples["human_review"].eq("확인 완료").sum()
)

scope_notes = (SUBMIT / "data_scope_notes.md").read_text(
    encoding="utf-8"
)


# 3. 최종 검토 보고서 작성
report = f"""# WI26 회귀분석 재현 및 기간 안정성 검토

## 1. 결론

기존 일별·5거래일·10거래일 분석의 비교 항목은 허용오차 내에서 재현됐다.
시간 순서를 유지한 확장형 3분할 검증에서는 모든 조건의
조합별 MSE 개선율 중앙값이 음수였다.

일부 조합에서 세 기간 모두 개선이 관찰됐지만,
금융지표 추가에 따른 전반적이고 안정적인 오차 개선은 확인하지 못했다.
개선 조합은 사후 탐색 결과로 보고하며 자동 채택하지 않는다.

## 2. 분석 대상과 방법

- 업종: WI26 26개
- 국내 금융지표: 6개
- 원자료 기간: 2023-11-06~2026-11-05
- 일별 수익률: 업종별 729개
- 시장 통제: KOSPI 단독 및 KOSPI·KOSDAQ
- 후보 지표는 한 번에 하나씩 추가
- 일별 및 비중첩 5·10거래일 분석
- 5·10거래일은 동시점과 직전 구간 지표를 사용하는 시차 분석
- 절편 포함 OLS
- 일별 HAC 5시차, 구간 HAC 1시차
- Bartlett 가중치, 소표본 보정, 정규근사
- 결측 수익률의 0 채우기와 이상치 삭제·윈저화 미적용

## 3. 자료와 기존 결과의 재현

배포 명세 60개 파일의 존재·해시·크기가 모두 일치했다.

일별 입력의 9개 수치 열을 포함된 수준값에서 재계산했다.
비중첩 5·10거래일 구간을 독립 생성하여 기존 입력과 대조했다.
신용스프레드와 장단기 금리차의 차감 방향도 확인했다.

전체 구간 종합 결과 및 KOSPI 구간 결과의 수치 24,492개를
대조한 결과 불일치가 없었다.
최초 일별 KOSPI 계수·표준오차·p값·BY156의 추가 대조도 통과했다.
이 비교 수들은 일부 중복 항목을 포함하므로 독립 검정 수를 의미하지 않는다.

실수 비교 허용오차는 atol=1e-8, rtol=1e-6을 적용했다.
표본 수는 정확히 일치하는지 비교했다.

BY156·BY624·BY780·BY1560·BY2340의 해당 검정군에서
보정 p값이 0.05 미만인 금융지표 계수는 없었다.
이는 관계가 전혀 없다는 입증과는 구별된다.

## 4. 시간 순서 검증 설계

요청서의 확장형 3분할 제안을 이번 실행 기준으로 적용했다.
이 분할이 기존 팀 합의였다고 주장하지 않는다.

각 분할의 학습 자료만으로 기준모형과 추가모형을 추정하고,
검증 구간에는 계수를 고정 적용했다.
검증 중 재추정이나 검증 성적에 따른 후보 변경은 하지 않았다.

기존 비중첩 구간의 시작점을 유지했으며,
분할 경계를 가로지르는 목표 수익률 구간은 제외하고 기록했다.
학습·검증 목표의 시간적 중복이 없음을 확인했다.

일별 검증은 123·120·124거래일,
5거래일 검증은 분할당 24구간,
10거래일 검증은 분할당 11구간이다.

전체 모형 조합은 {len(all_cv_pair_summary):,}개,
분할별 평가는 {len(all_cv_metrics):,}개이다.
검증 계산값 {len(all_cv_fitted):,}행은 모형별 기록이며
서로 독립적인 거래일 수가 아니다.

## 5. 평가 결과

개선율은 다음과 같이 계산했다.

`100 × (1 − 추가모형 검증MSE / 기준모형 검증MSE)`

통합 오차는 분할별 표본 수를 반영해 계산했다.
분할별 개선율의 단순 평균으로 통합 성적을 대신하지 않았다.

{markdown_table(report_overview)}

모든 조건의 중앙값이 음수였다.
10거래일 동시점·두 시장 통제는 평균이 양수였지만
중앙값은 음수였으므로 전체적인 개선으로 해석하지 않았다.

개선된 기간 수·최악의 기간·통합 MSE·RMSE·MAE와
분할별 계수·HAC 표준오차·신뢰구간을 별도 표에 기록했다.
학습 계수의 미보정 p값을 후보 채택 기준으로 사용하지 않았다.

## 6. 진단과 그림

모든 학습 추가모형에 대해 랭크·조건수·각 설명변수 VIF,
잔차 자기상관·이분산·Cook 거리·레버리지를 확인했다.

일부 모형에서 자기상관·이분산 참고표시와
영향이 큰 학습 관측값이 나타났다.
진단 p값은 미보정 참고값이며 자동 탈락이나 삭제 기준으로 사용하지 않았다.

일별 자기상관 진단은 5시차,
구간 진단은 구간 단위 1시차를 사용했다.

최초 재현 조합인 에너지 × 국고채 3년을 고정하여
학습·검증 산점도와 잔차 그림을 생성했다.
이 예시 그림을 전체 조합의 결과로 일반화하지 않는다.

## 7. 자료 감사와 제한

{markdown_table(audit_status)}

대표 날짜의 계산 예시 27개가 입력과 일치했다.
파일에 확인 완료로 기록된 사람의 직접 확인 항목은 {human_checked}개이다.
프로그램 재계산을 사람의 손계산 확인으로 표시하지 않았다.

{scope_notes}

## 8. 해석 범위와 다음 확인

목표 기간 시장 수익률과 금융지표 변화를 사용하므로
실행 가능한 미래 예측이나 투자전략 성과, 인과효과를 검증한 것이 아니다.

전체 기간 결과를 확인한 뒤 수행한 사후 기간 안정성 점검이다.
겹치는 학습 구간의 세 분할을 독립된 세 실험으로 취급하지 않는다.

10거래일 검증은 표본이 작아 탐색적 결과로 해석한다.
향후 확증이 필요하면 규칙을 고정하고 아직 살펴보지 않은 자료를 확보한다.
미검증 원자료와 발표시점은 추가 자료를 확보한 뒤 확인한다.
"""

(SUBMIT / "review.md").write_text(
    report,
    encoding="utf-8"
)


# 4. 재현 차이표와 분할표를 제출 폴더로 복사
support_files = [
    "transformation_audit.csv",
    "single_pair_reproduction_diff.csv",
    "daily_reproduction_diff.csv",
    "daily_kospi_by156_reproduction_diff.csv",
    "block_reproduction_audit.csv",
    "all_horizons_reproduction_diff.csv",
    "split_plan.csv",
    "splits.csv",
    "block_splits.csv",
    "excluded_incomplete_blocks.csv",
    "excluded_boundary_blocks.csv",
    "model_diagnostics.csv",
    "full_model_vif.csv",
    "influential_training_dates.csv",
    "block_model_diagnostics.csv",
    "block_full_model_vif.csv",
    "block_influential_training_intervals.csv"
]

for filename in support_files:
    source = OUT / filename
    assert source.is_file(), f"필요한 결과 파일 없음: {filename}"
    shutil.copy2(source, SUBMIT / filename)

# 전체 구간 재현 차이표를 요청서의 대표 파일명으로도 제공
shutil.copy2(
    OUT / "all_horizons_reproduction_diff.csv",
    SUBMIT / "reproduction_diff.csv"
)


# 5. 전달용 설명서 작성
readme = """# WI26 회귀분석 검증 제출 자료

## 읽는 순서

1. review.md: 결론·방법·한계
2. cv_overall_summary.csv: 전체 조건 요약
3. cv_pair_summary.csv: 전체 모형 조합의 통합 성적
4. cv_metrics.csv: 분할별 평가 성적
5. coefficient_stability.csv: 학습 구간별 금융지표 계수
6. reproduction_diff.csv: 기존 전체 구간 결과와의 대조
7. data_audit.csv: 자료 감사 결과
8. figures/: 진단 그림

## 실행과 추적

- 실제 Python·패키지 버전은 environment.json과 requirements.txt 참조
- 입력의 지문·출처·기간·단위는 input_manifest.csv 참조
- 원래 배포 파일의 해시 검사는 handoff_hash_audit.csv 참조
- 일별 실제 분할은 splits.csv, 구간 분할은 block_splits.csv 참조
- 제외 사유는 excluded_incomplete_blocks.csv와
  excluded_boundary_blocks.csv 참조

## 완료와 제한

일별·5일·10일 회귀 재현, BY 보정 대조,
확장형 시간 분할 평가 및 기본 진단을 수행했다.

원본 엑셀 추출·외부 가격의 전기간 대조·실제 발표시점·
공식 거래일 완전성은 미검증이다.
사람의 직접 계산 확인 여부는 calculation_examples.csv 참조.

현재는 결과·설명서 정리 단계이다.
실행 노트북과 입력자료 포함 및 새 커널 전체 재실행 확인은
최종 포장 단계에서 별도로 확인해야 한다.
"""

(SUBMIT / "README.md").write_text(
    readme,
    encoding="utf-8"
)

print("【보고서 저장 완료】")
print(SUBMIT / "review.md")
print(SUBMIT / "README.md")
print("\n사람의 직접 확인 완료 항목 수:", human_checked)
print("보조 결과 파일 복사 수:", len(support_files))
print("\n아직 ZIP 생성과 전체 재실행 확인은 하지 않았습니다.")

In [ ]:
# 제출 패키지 생성
import ast
import json
import shutil
import sys
from datetime import datetime

# 1. 저장한 노트북 찾기
notebook_name = "wi26_reproduction_and_time_series_validation.ipynb"
search_root = ROOT.parent

candidates = [
    path
    for path in search_root.rglob(notebook_name)
    if ".ipynb_checkpoints" not in path.parts
    and not any(
        part.startswith("wi26_validation_delivery_")
        for part in path.parts
    )
]

print("찾은 노트북:")
for path in candidates:
    print(path)

assert len(candidates) == 1, (
    "노트북이 정확히 하나여야 합니다. "
    "파일을 저장했는지, 이름이 맞는지 확인하세요."
)

notebook_path = candidates[0]
notebook = json.loads(
    notebook_path.read_text(encoding="utf-8-sig")
)


# 2. 최종 보고서 작성 셀까지 코드 추출
# 현재 포장 셀과 이후 셀은 실행 코드에 포함하지 않음
code_cells = []
saved_cells = []
found_report = False

for cell in notebook["cells"]:
    source = cell.get("source", "")
    if isinstance(source, list):
        source = "".join(source)

    if source.lstrip().startswith("# 제출 패키지 생성"):
        break

    saved_cells.append(cell)

    if cell.get("cell_type") != "code":
        continue

    if not source.strip():
        continue

    # 설치 명령만 있는 셀은 실행 스크립트에서 제외
    meaningful_lines = [
        line.strip()
        for line in source.splitlines()
        if line.strip() and not line.lstrip().startswith("#")
    ]

    if meaningful_lines and all(
        line.startswith("%pip ") or line.startswith("!pip ")
        for line in meaningful_lines
    ):
        continue

    assert not any(
        line.startswith(("%", "!"))
        for line in meaningful_lines
    ), "자동 변환할 수 없는 노트북 명령이 있습니다."

    # 시작 셀의 개인 절대경로를 패키지 내부 경로로 교체
    if (
        "daily_inputs_with_kosdaq.csv" in source
        and "rglob" in source
        and "matches" in source
    ):
        source = """
from pathlib import Path

ROOT = PROJECT_DIR / "handoff"
INPUT = ROOT / "data/dual/daily_inputs_with_kosdaq.csv"

assert INPUT.is_file(), "패키지의 입력 파일을 찾을 수 없습니다."
assert (ROOT / "docs/VALIDATION_REQUEST.md").is_file()

print("전달자료 위치:", ROOT)
"""

    tree = ast.parse(source)

    # 실행 위치에 관계없이 결과를 패키지 내부에 저장
    for node in tree.body:
        if isinstance(node, ast.Assign):
            if any(
                isinstance(target, ast.Name)
                and target.id == "OUT"
                for target in node.targets
            ):
                node.value = ast.parse(
                    'PROJECT_DIR / "results"',
                    mode="eval"
                ).body

    tree = ast.fix_missing_locations(tree)
    code_cells.append(ast.unparse(tree))

    if source.lstrip().startswith("# 최종 보고서 작성"):
        found_report = True
        break

assert found_report, (
    "저장된 노트북에서 최종 보고서 작성 셀을 찾지 못했습니다. "
    "직전 보고서 셀을 실행하고 Ctrl+S로 저장한 뒤 다시 실행하세요."
)


# 3. 실행 스크립트 구성
bootstrap = """
from pathlib import Path
import matplotlib

matplotlib.use("Agg")

PROJECT_DIR = Path(__file__).resolve().parent
"""

script_text = bootstrap + "\n\n".join(
    f"# 실행 단계 {number}\n{source}"
    for number, source in enumerate(code_cells, start=1)
)

# 문법 검사: 실제 분석 재실행과는 별개
compile(script_text, "run_validation.py", "exec")


# 4. 새 전달 폴더 생성
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")

PACKAGE = OUT.parent / f"wi26_validation_delivery_{stamp}"
PACKAGE.mkdir(exist_ok=False)

script_path = PACKAGE / "run_validation.py"
script_path.write_text(script_text, encoding="utf-8")

# 원래 입력자료를 상대경로 구조로 보존
handoff_target = PACKAGE / "handoff"
handoff_target.mkdir()

for directory in ["data", "docs", "reference", "scripts", "verification"]:
    source = ROOT / directory
    assert source.is_dir(), f"전달자료 폴더 없음: {directory}"
    shutil.copytree(source, handoff_target / directory)

for filename in ["README.md", "FILE_MANIFEST.csv"]:
    shutil.copy2(ROOT / filename, handoff_target / filename)

# 현재 제출 결과 보존
shutil.copytree(SUBMIT, PACKAGE / "submission")

# 현재 실행 결과도 함께 보존
shutil.copytree(OUT, PACKAGE / "results")


# 5. 포장 셀을 제외한 노트북 사본 저장
notebook_copy = dict(notebook)
notebook_copy["cells"] = saved_cells

(PACKAGE / notebook_name).write_text(
    json.dumps(notebook_copy, ensure_ascii=False, indent=1),
    encoding="utf-8"
)


# 6. 패키지 실행 안내 작성
delivery_readme = """# WI26 회귀분석 검증 전달 자료

## 먼저 읽을 파일

- submission/review.md: 최종 검토 보고서
- submission/cv_overall_summary.csv: 전체 검증 요약
- submission/README.md: 결과 파일 안내
- submission/figures/: 산점도와 잔차 그림

## 폴더 구성

- handoff/: 팀원이 전달한 입력과 기존 결과
- submission/: 제출용 결과·환경·감사 기록
- results/: 단계별 계산 결과
- run_validation.py: 입력부터 보고서까지 실행하는 코드
- 주피터 노트북: 원래 작업 과정과 출력 기록

## 재실행

1. submission/environment.json에서 Python 버전을 확인한다.
2. 필요한 환경에서 다음 명령으로 패키지를 설치한다.

   python -m pip install -r submission/requirements.txt

3. 다음 명령으로 실행한다.

   python run_validation.py

실행 스크립트는 자기 위치를 기준으로 입력과 출력 경로를 정한다.
노트북 사본에는 작업 당시 경로가 남을 수 있으므로,
다른 PC에서는 run_validation.py 사용을 권장한다.

현재 패키지는 구성 및 문법 검사를 마친 상태이다.
깨끗한 프로세스에서의 전체 재실행 결과는 별도 기록으로 확인해야 한다.
"""

(PACKAGE / "README.md").write_text(
    delivery_readme,
    encoding="utf-8"
)

# 다음 셀에서 사용할 패키지 위치 기록
(OUT.parent / "latest_delivery_path.txt").write_text(
    str(PACKAGE.resolve()),
    encoding="utf-8"
)

print("\n【전달 폴더 준비 완료】")
print("폴더:", PACKAGE)
print("추출한 코드 셀 수:", len(code_cells))
print("실행 코드:", script_path)
print("문법 검사: 통과")
print("\n아직 전체 재실행과 ZIP 생성은 하지 않았습니다.")